# NBA Lineup Replay and Stint Prototype

Consolidated notebook with the working lessons from `01_pbp_exploration.ipynb`.

Workflow:
1. Load cached play-by-play
2. Parse substitution rows and build same-clock substitution batches
3. Build grouped event timeline
4. Replay substitutions with strict validation and period-start lineup resets
5. Surface first failing substitution per period to fix seeds
6. Build first `fact_lineup_stint` prototype from grouped timestamps

This version uses **player names** for easier manual debugging.

In [1]:
# Optional one-time setup (safe to re-run)
%pip install -q -r ../requirements.txt
# Fallback if requirements file is missing packages:
# %pip install -q nba_api pandas

Note: you may need to restart the kernel to use updated packages.


In [2]:
import re
import sys
from pathlib import Path

import pandas as pd

# Ensure project root is on sys.path even when notebook cwd is notebooks/
_cwd = Path.cwd().resolve()
_repo_root = _cwd.parent if _cwd.name == "notebooks" else _cwd
if str(_repo_root) not in sys.path:
    sys.path.insert(0, str(_repo_root))

from src.ingestion.pbp_json import load_pbp_actions_dataframe

GAME_ID = "0042500111"

pbp_df = load_pbp_actions_dataframe(GAME_ID)
pbp_df = pbp_df.sort_values("actionNumber", kind="mergesort").reset_index(drop=True)

# Parse clock to seconds remaining in current period

def clock_to_seconds_remaining(clock_str):
    if pd.isna(clock_str):
        return None
    m = re.match(r"PT(?:(\d+)M)?([0-9.]+)S", str(clock_str))
    if not m:
        return None
    minutes = int(m.group(1)) if m.group(1) else 0
    seconds = float(m.group(2))
    return minutes * 60 + seconds

pbp_df["clock_seconds_remaining"] = pbp_df["clock"].apply(clock_to_seconds_remaining)

print("Loaded actions:", len(pbp_df))
display(pbp_df.head(5))

Loaded actions: 485


,actionNumber,clock,period,teamId,teamTricode,personId,playerName,playerNameI,xLegacy,yLegacy,...,scoreAway,pointsTotal,location,description,actionType,subType,videoAvailable,shotValue,actionId,clock_seconds_remaining
0,2,PT12M00.00S,1,0,,0,,,0,0,...,0,0,,Start of 1st Period (1:11 PM EST),period,start,1,0,1,720.0
1,4,PT12M00.00S,1,1610612738,BOS,1629674,Queta,N. Queta,0,0,...,,0,h,Jump Ball Queta vs. Bona: Tip to Edgecombe,Jump Ball,,1,0,2,720.0
2,7,PT11M48.00S,1,1610612755,PHI,202331,George,P. George,123,234,...,,0,v,MISS George 26' 3PT Jump Shot,Missed Shot,Jump Shot,1,3,3,708.0
3,8,PT11M47.00S,1,1610612738,BOS,1628369,Tatum,J. Tatum,0,0,...,,0,h,Tatum REBOUND (Off:0 Def:1),Rebound,Unknown,1,0,4,707.0
4,9,PT11M38.00S,1,1610612755,PHI,1641737,Bona,A. Bona,0,0,...,,0,v,Bona S.FOUL (P1.T1) (S.Foster),Foul,Shooting,1,0,5,698.0


In [3]:
# Build substitution events with both outgoing sources:
# - player_out_name_api (from playerName / personId actor on sub row)
# - player_out_name_text (from SUB: X FOR Y text)

SUB_PATTERN = re.compile(r"^SUB:\s*(?P<player_in>.+?)\s+FOR\s+(?P<player_out>.+?)\s*$", re.IGNORECASE)

sub_df = pbp_df[pbp_df["actionType"].eq("Substitution")].copy()
parsed = sub_df["description"].astype(str).str.extract(SUB_PATTERN)

sub_df["player_in_name"] = parsed["player_in"].str.strip()
sub_df["player_out_name_text"] = parsed["player_out"].str.strip()
sub_df["player_out_name_api"] = sub_df["playerName"].astype(str).str.strip()
sub_df["player_out_id"] = sub_df["personId"]

sub_event_cols = [
    "actionNumber", "actionId", "period", "clock", "teamId", "teamTricode",
    "player_out_id", "player_out_name_api", "player_in_name", "player_out_name_text",
    "description"
]

sub_events_clean = (
    sub_df[sub_event_cols]
    .sort_values(["period", "clock", "actionNumber"], ascending=[True, False, True])
    .reset_index(drop=True)
)

sub_groups_df = (
    sub_events_clean.groupby(["period", "clock"], sort=False)
    .apply(lambda g: pd.Series({"n_subs": len(g), "substitutions": g.to_dict("records")}))
    .reset_index()
)

sub_group_lookup = {
    (row["period"], row["clock"]): row["substitutions"]
    for _, row in sub_groups_df.iterrows()
}

print("Substitution rows:", len(sub_df))
print("Sub batches:", len(sub_groups_df))
display(sub_groups_df.head(10))

Substitution rows: 45
Sub batches: 31


,period,clock,n_subs,substitutions
0,1,PT10M23.00S,1,"[{'actionNumber': 24, 'actionId': 17, 'teamId'..."
1,1,PT08M39.00S,1,"[{'actionNumber': 49, 'actionId': 37, 'teamId'..."
2,1,PT04M50.00S,3,"[{'actionNumber': 91, 'actionId': 68, 'teamId'..."
3,1,PT04M39.00S,1,"[{'actionNumber': 106, 'actionId': 75, 'teamId..."
4,1,PT03M49.00S,1,"[{'actionNumber': 120, 'actionId': 86, 'teamId..."
5,1,PT02M39.00S,1,"[{'actionNumber': 133, 'actionId': 97, 'teamId..."
6,1,PT00M27.30S,2,"[{'actionNumber': 157, 'actionId': 115, 'teamI..."
7,2,PT08M43.00S,3,"[{'actionNumber': 221, 'actionId': 157, 'teamI..."
8,2,PT07M16.00S,2,"[{'actionNumber': 241, 'actionId': 171, 'teamI..."
9,2,PT06M50.00S,1,"[{'actionNumber': 250, 'actionId': 176, 'teamI..."


In [4]:
# Group all events by timestamp (period, clock)

event_groups_df = (
    pbp_df.groupby(["period", "clock", "clock_seconds_remaining"], sort=False)
    .agg(
        n_events=("actionNumber", "count"),
        action_numbers=("actionNumber", list),
        action_types=("actionType", list),
    )
    .reset_index()
    .sort_values(["period", "clock_seconds_remaining"], ascending=[True, False])
    .reset_index(drop=True)
)

sub_keys = set(zip(sub_groups_df["period"], sub_groups_df["clock"]))
event_groups_df["has_substitution"] = event_groups_df.apply(
    lambda row: (row["period"], row["clock"]) in sub_keys,
    axis=1,
)

print("Event timestamp groups:", len(event_groups_df))
display(event_groups_df.head(20))

Event timestamp groups: 329


,period,clock,clock_seconds_remaining,n_events,action_numbers,action_types,has_substitution
0,1,PT12M00.00S,720.0,2,"[2, 4]","[period, Jump Ball]",False
1,1,PT11M48.00S,708.0,1,[7],[Missed Shot],False
2,1,PT11M47.00S,707.0,1,[8],[Rebound],False
3,1,PT11M38.00S,698.0,4,"[9, 11, 12, 13]","[Foul, Free Throw, Free Throw, Rebound]",False
4,1,PT11M24.00S,684.0,1,[14],[Missed Shot],False
5,1,PT11M23.00S,683.0,1,[15],[Rebound],False
6,1,PT11M15.00S,675.0,1,[16],[Made Shot],False
7,1,PT10M57.00S,657.0,1,[18],[Made Shot],False
8,1,PT10M29.00S,629.0,1,[19],[Missed Shot],False
9,1,PT10M28.00S,628.0,1,[20],[Rebound],False


In [5]:
# Manual period-start lineup seeds (player names).
# Update these as validation discovers mismatches.

home_team_id = 1610612738
away_team_id = 1610612755

period_start_lineups = {
    1: {
        "home": {"Hauser", "Tatum", "Queta", "Brown", "White"},
        "away": {"George", "Oubre Jr.", "Bona", "Edgecombe", "Maxey"},
    },
    2: {
        "home": {"Pritchard", "Walsh", "Brown", "White", "Scheierman"},
        "away": {"George", "Grimes", "Barlow", "Edwards", "Drummond"},
    },
    3: {
        "home": {"Hauser", "Tatum", "Vucevic", "Brown", "White"},
        "away": {"George", "Oubre Jr.", "Bona", "Edgecombe", "Maxey"},
    },
    4: {
        "home": {"Hauser", "Tatum", "Vucevic", "Brown", "White"},
        "away": {"George", "Oubre Jr.", "Drummond", "Edgecombe", "Maxey"},
    },
}

for p, s in period_start_lineups.items():
    assert len(s["home"]) == 5, f"P{p} home size != 5"
    assert len(s["away"]) == 5, f"P{p} away size != 5"

print("Period seeds loaded for:", sorted(period_start_lineups.keys()))

Period seeds loaded for: [1, 2, 3, 4]


In [6]:
# Strict batch applier + period reset helper

def reset_lineup_state_for_period(period, period_start_lineups):
    if period not in period_start_lineups:
        raise ValueError(f"No lineup seed for period {period}")
    home = set(period_start_lineups[period]["home"])
    away = set(period_start_lineups[period]["away"])
    if len(home) != 5 or len(away) != 5:
        raise ValueError(f"P{period} lineup seed must be 5 players each")
    print(f"\n[RESET] P{period}")
    print(" HOME:", sorted(home))
    print(" AWAY:", sorted(away))
    return home, away


def apply_substitution_batch_strict(
    home_lineup: set[str],
    away_lineup: set[str],
    sub_batch: list[dict],
    home_team_id: int,
    away_team_id: int,
):
    """
    Strict substitution application for notebook validation.
    Rules:
    - outgoing player must already be on floor
    - incoming player must not already be on floor

    Returns updated lineups + rich debug rows.
    """
    home = set(home_lineup)
    away = set(away_lineup)
    debug_rows = []

    for sub in sub_batch:
        team_id = int(sub["teamId"])
        player_in = str(sub["player_in_name"]).strip()
        # Prefer API outgoing name; fallback keeps older parsed objects usable.
        player_out = str(
            sub.get("player_out_name_api")
            or sub.get("player_out_name")
            or sub.get("player_out_name_text")
            or ""
        ).strip()

        if team_id == home_team_id:
            team_label = "HOME"
            lineup = home
        elif team_id == away_team_id:
            team_label = "AWAY"
            lineup = away
        else:
            debug_rows.append({
                "team": "UNKNOWN",
                "team_id": team_id,
                "player_out": player_out,
                "player_in": player_in,
                "outgoing_found": False,
                "incoming_already_present": False,
                "sub_succeeded": False,
                "error_type": "unknown_team",
                "message": f"Unknown teamId={team_id}",
                "description": sub.get("description"),
                "actionNumber": sub.get("actionNumber"),
            })
            continue

        outgoing_found = player_out in lineup
        incoming_already_present = player_in in lineup
        sub_succeeded = outgoing_found and (not incoming_already_present)

        error_type = None
        msg = "ok"
        if not outgoing_found and incoming_already_present:
            error_type = "out_missing_and_in_already_present"
            msg = f"{team_label}: OUT '{player_out}' missing, IN '{player_in}' already on floor"
        elif not outgoing_found:
            error_type = "outgoing_not_found"
            msg = f"{team_label}: OUT '{player_out}' not found on floor"
        elif incoming_already_present:
            error_type = "incoming_already_present"
            msg = f"{team_label}: IN '{player_in}' already on floor"

        if sub_succeeded:
            lineup.remove(player_out)
            lineup.add(player_in)
        else:
            print(f"[WARN] {msg}")

        debug_rows.append({
            "team": team_label,
            "team_id": team_id,
            "player_out": player_out,
            "player_in": player_in,
            "outgoing_found": outgoing_found,
            "incoming_already_present": incoming_already_present,
            "sub_succeeded": sub_succeeded,
            "error_type": error_type,
            "message": msg,
            "description": sub.get("description"),
            "actionNumber": sub.get("actionNumber"),
        })

    return home, away, debug_rows

In [7]:
# Replay all substitution batches with strict logging

eg = event_groups_df.copy().sort_values(
    ["period", "clock_seconds_remaining"],
    ascending=[True, False],
    kind="mergesort",
).reset_index(drop=True)

current_period = None
home_state, away_state = None, None
validation_rows = []

for _, row in eg.iterrows():
    period = int(row["period"])
    clock = row["clock"]
    has_sub = bool(row["has_substitution"])

    if period != current_period:
        current_period = period
        home_state, away_state = reset_lineup_state_for_period(period, period_start_lineups)

    if not has_sub:
        continue

    batch = sub_group_lookup.get((period, clock), [])
    home_before = sorted(home_state)
    away_before = sorted(away_state)

    home_state, away_state, sub_debug = apply_substitution_batch_strict(
        home_state, away_state, batch, home_team_id, away_team_id
    )

    home_ok = len(home_state) == 5
    away_ok = len(away_state) == 5

    for d in sub_debug:
        validation_rows.append({
            "period": period,
            "clock": clock,
            "team": d["team"],
            "team_id": d["team_id"],
            "player_out": d["player_out"],
            "player_in": d["player_in"],
            "outgoing_found": d["outgoing_found"],
            "incoming_already_present": d["incoming_already_present"],
            "sub_succeeded": d["sub_succeeded"],
            "error_type": d["error_type"],
            "message": d["message"],
            "home_size_ok": home_ok,
            "away_size_ok": away_ok,
            "home_lineup_before": home_before,
            "away_lineup_before": away_before,
            "home_lineup_after": sorted(home_state),
            "away_lineup_after": sorted(away_state),
            "description": d.get("description"),
            "actionNumber": d.get("actionNumber"),
        })

validation_df = pd.DataFrame(validation_rows)

print("\nReplay complete.")
print("Sub rows logged:", len(validation_df))
if len(validation_df):
    print("Sub failures:", int((~validation_df["sub_succeeded"]).sum()))
    print("Home size failures:", int((~validation_df["home_size_ok"]).sum()))
    print("Away size failures:", int((~validation_df["away_size_ok"]).sum()))

display(validation_df.head(30))


[RESET] P1
 HOME: ['Brown', 'Hauser', 'Queta', 'Tatum', 'White']
 AWAY: ['Bona', 'Edgecombe', 'George', 'Maxey', 'Oubre Jr.']
[WARN] HOME: OUT 'Vučević' not found on floor

[RESET] P2
 HOME: ['Brown', 'Pritchard', 'Scheierman', 'Walsh', 'White']
 AWAY: ['Barlow', 'Drummond', 'Edwards', 'George', 'Grimes']
[WARN] HOME: OUT 'Garza' missing, IN 'White' already on floor
[WARN] HOME: OUT 'Hauser' not found on floor
[WARN] HOME: OUT 'Vučević' not found on floor

[RESET] P3
 HOME: ['Brown', 'Hauser', 'Tatum', 'Vucevic', 'White']
 AWAY: ['Bona', 'Edgecombe', 'George', 'Maxey', 'Oubre Jr.']
[WARN] HOME: OUT 'Queta' missing, IN 'Vucevic' already on floor
[WARN] AWAY: OUT 'Drummond' not found on floor
[WARN] AWAY: IN 'Bona' already on floor
[WARN] HOME: OUT 'Vučević' not found on floor
[WARN] AWAY: IN 'Oubre Jr.' already on floor
[WARN] HOME: OUT 'Garza' not found on floor

[RESET] P4
 HOME: ['Brown', 'Hauser', 'Tatum', 'Vucevic', 'White']
 AWAY: ['Drummond', 'Edgecombe', 'George', 'Maxey', 'Oub

,period,clock,team,team_id,player_out,player_in,outgoing_found,incoming_already_present,sub_succeeded,error_type,message,home_size_ok,away_size_ok,home_lineup_before,away_lineup_before,home_lineup_after,away_lineup_after,description,actionNumber
0,1,PT10M23.00S,AWAY,1610612755,Bona,Drummond,True,False,True,NaN,ok,True,True,"[Brown, Hauser, Queta, Tatum, White]","[Bona, Edgecombe, George, Maxey, Oubre Jr.]","[Brown, Hauser, Queta, Tatum, White]","[Drummond, Edgecombe, George, Maxey, Oubre Jr.]",SUB: Drummond FOR Bona,24
1,1,PT08M39.00S,HOME,1610612738,Queta,Vucevic,True,False,True,NaN,ok,True,True,"[Brown, Hauser, Queta, Tatum, White]","[Drummond, Edgecombe, George, Maxey, Oubre Jr.]","[Brown, Hauser, Tatum, Vucevic, White]","[Drummond, Edgecombe, George, Maxey, Oubre Jr.]",SUB: Vucevic FOR Queta,49
2,1,PT04M50.00S,HOME,1610612738,Hauser,Pritchard,True,False,True,NaN,ok,True,True,"[Brown, Hauser, Tatum, Vucevic, White]","[Drummond, Edgecombe, George, Maxey, Oubre Jr.]","[Brown, Pritchard, Tatum, Vucevic, White]","[Barlow, Drummond, George, Grimes, Maxey]",SUB: Pritchard FOR Hauser,91
3,1,PT04M50.00S,AWAY,1610612755,Oubre Jr.,Grimes,True,False,True,NaN,ok,True,True,"[Brown, Hauser, Tatum, Vucevic, White]","[Drummond, Edgecombe, George, Maxey, Oubre Jr.]","[Brown, Pritchard, Tatum, Vucevic, White]","[Barlow, Drummond, George, Grimes, Maxey]",SUB: Grimes FOR Oubre Jr.,92
4,1,PT04M50.00S,AWAY,1610612755,Edgecombe,Barlow,True,False,True,NaN,ok,True,True,"[Brown, Hauser, Tatum, Vucevic, White]","[Drummond, Edgecombe, George, Maxey, Oubre Jr.]","[Brown, Pritchard, Tatum, Vucevic, White]","[Barlow, Drummond, George, Grimes, Maxey]",SUB: Barlow FOR Edgecombe,93
5,1,PT04M39.00S,AWAY,1610612755,Drummond,Oubre Jr.,True,False,True,NaN,ok,True,True,"[Brown, Pritchard, Tatum, Vucevic, White]","[Barlow, Drummond, George, Grimes, Maxey]","[Brown, Pritchard, Tatum, Vucevic, White]","[Barlow, George, Grimes, Maxey, Oubre Jr.]",SUB: Oubre Jr. FOR Drummond,106
6,1,PT03M49.00S,HOME,1610612738,Brown,Walsh,True,False,True,NaN,ok,True,True,"[Brown, Pritchard, Tatum, Vucevic, White]","[Barlow, George, Grimes, Maxey, Oubre Jr.]","[Pritchard, Tatum, Vucevic, Walsh, White]","[Barlow, George, Grimes, Maxey, Oubre Jr.]",SUB: Walsh FOR Brown,120
7,1,PT02M39.00S,AWAY,1610612755,George,Edwards,True,False,True,NaN,ok,True,True,"[Pritchard, Tatum, Vucevic, Walsh, White]","[Barlow, George, Grimes, Maxey, Oubre Jr.]","[Pritchard, Tatum, Vucevic, Walsh, White]","[Barlow, Edwards, Grimes, Maxey, Oubre Jr.]",SUB: Edwards FOR George,133
8,1,PT00M27.30S,HOME,1610612738,Vučević,Brown,False,False,False,outgoing_not_found,HOME: OUT 'Vučević' not found on floor,True,True,"[Pritchard, Tatum, Vucevic, Walsh, White]","[Barlow, Edwards, Grimes, Maxey, Oubre Jr.]","[Pritchard, Tatum, Vucevic, Walsh, White]","[Edwards, George, Grimes, Maxey, Oubre Jr.]",SUB: Brown FOR Vucevic,157
9,1,PT00M27.30S,AWAY,1610612755,Barlow,George,True,False,True,NaN,ok,True,True,"[Pritchard, Tatum, Vucevic, Walsh, White]","[Barlow, Edwards, Grimes, Maxey, Oubre Jr.]","[Pritchard, Tatum, Vucevic, Walsh, White]","[Edwards, George, Grimes, Maxey, Oubre Jr.]",SUB: George FOR Barlow,158


In [9]:
# First failing substitution per period + seed correction suggestions

if validation_df.empty:
    print("No substitution validation rows found.")
else:
    fail_mask = (
        (~validation_df["sub_succeeded"])
        | (~validation_df["home_size_ok"])
        | (~validation_df["away_size_ok"])
    )
    fail_df = validation_df[fail_mask].copy().sort_values(
        ["period", "clock", "actionNumber"],
        ascending=[True, False, True],
    )

    print("Failing rows:", len(fail_df))
    display(fail_df[[
        "period", "clock", "team", "player_out", "player_in",
        "outgoing_found", "incoming_already_present",
        "sub_succeeded", "home_size_ok", "away_size_ok", "error_type"
    ]])

    first_fail_per_period = fail_df.groupby("period", as_index=False).first()
    print("\nFirst failing substitution per period:")
    display(first_fail_per_period[[
        "period", "clock", "team", "player_out", "player_in",
        "error_type", "home_lineup_before", "away_lineup_before"
    ]])

    suggestions = []
    for _, r in first_fail_per_period.iterrows():
        suggestions.append({
            "period": int(r["period"]),
            "suggested_home_seed": r["home_lineup_before"],
            "suggested_away_seed": r["away_lineup_before"],
            "based_on": f"P{int(r['period'])} {r['clock']} {r['team']} {r['player_out']} -> {r['player_in']}",
        })

    suggestions_df = pd.DataFrame(suggestions)
    print("\nSuggested period seed fixes (manual review):")
    display(suggestions_df)

Failing rows: 14


,period,clock,team,player_out,player_in,outgoing_found,incoming_already_present,sub_succeeded,home_size_ok,away_size_ok,error_type
8,1,PT00M27.30S,HOME,Vučević,Brown,False,False,False,True,True,outgoing_not_found
10,2,PT08M43.00S,HOME,Garza,White,False,True,False,True,True,out_missing_and_in_already_present
13,2,PT07M16.00S,HOME,Hauser,Tatum,False,False,False,True,True,outgoing_not_found
19,2,PT02M10.00S,HOME,Vučević,Garza,False,False,False,True,True,outgoing_not_found
23,3,PT10M25.00S,HOME,Queta,Vucevic,False,True,False,True,True,out_missing_and_in_already_present
25,3,PT04M33.00S,AWAY,Drummond,Grimes,False,False,False,True,True,outgoing_not_found
26,3,PT04M33.00S,AWAY,Oubre Jr.,Bona,True,True,False,True,True,incoming_already_present
28,3,PT03M36.00S,HOME,Vučević,Garza,False,False,False,True,True,outgoing_not_found
30,3,PT02M10.00S,AWAY,Edgecombe,Oubre Jr.,True,True,False,True,True,incoming_already_present
31,3,PT00M43.30S,HOME,Garza,Hauser,False,False,False,True,True,outgoing_not_found



First failing substitution per period:


,period,clock,team,player_out,player_in,error_type,home_lineup_before,away_lineup_before
0,1,PT00M27.30S,HOME,Vučević,Brown,outgoing_not_found,"[Pritchard, Tatum, Vucevic, Walsh, White]","[Barlow, Edwards, Grimes, Maxey, Oubre Jr.]"
1,2,PT08M43.00S,HOME,Garza,White,out_missing_and_in_already_present,"[Brown, Pritchard, Scheierman, Walsh, White]","[Barlow, Drummond, Edwards, George, Grimes]"
2,3,PT10M25.00S,HOME,Queta,Vucevic,out_missing_and_in_already_present,"[Brown, Hauser, Tatum, Vucevic, White]","[Bona, Edgecombe, George, Maxey, Oubre Jr.]"
3,4,PT09M16.00S,AWAY,Edwards,Drummond,out_missing_and_in_already_present,"[Brown, Hauser, Tatum, Vucevic, White]","[Drummond, Edgecombe, George, Maxey, Oubre Jr.]"



Suggested period seed fixes (manual review):


,period,suggested_home_seed,suggested_away_seed,based_on
0,1,"[Pritchard, Tatum, Vucevic, Walsh, White]","[Barlow, Edwards, Grimes, Maxey, Oubre Jr.]",P1 PT00M27.30S HOME Vučević -> Brown
1,2,"[Brown, Pritchard, Scheierman, Walsh, White]","[Barlow, Drummond, Edwards, George, Grimes]",P2 PT08M43.00S HOME Garza -> White
2,3,"[Brown, Hauser, Tatum, Vucevic, White]","[Bona, Edgecombe, George, Maxey, Oubre Jr.]",P3 PT10M25.00S HOME Queta -> Vucevic
3,4,"[Brown, Hauser, Tatum, Vucevic, White]","[Drummond, Edgecombe, George, Maxey, Oubre Jr.]",P4 PT09M16.00S AWAY Edwards -> Drummond


In [10]:
# Helper functions for first fact_lineup_stint prototype

def get_score_snapshot_for_group(pbp_df, period, clock):
    rows = pbp_df[(pbp_df["period"] == period) & (pbp_df["clock"] == clock)].copy()
    if rows.empty:
        return None, None
    rows = rows.sort_values("actionNumber", kind="mergesort")
    rows["scoreHome"] = pd.to_numeric(rows["scoreHome"], errors="coerce").ffill()
    rows["scoreAway"] = pd.to_numeric(rows["scoreAway"], errors="coerce").ffill()
    sh = rows["scoreHome"].iloc[-1]
    sa = rows["scoreAway"].iloc[-1]
    return (None if pd.isna(sh) else int(sh), None if pd.isna(sa) else int(sa))


def append_stint_row(
    out_rows,
    period,
    start_clock,
    end_clock,
    start_sec,
    end_sec,
    home_lineup,
    away_lineup,
    score_home_start,
    score_away_start,
    score_home_end,
    score_away_end,
    ended_by_substitution,
):
    duration = max(0.0, float(start_sec) - float(end_sec))
    out_rows.append({
        "period": int(period),
        "start_clock": start_clock,
        "end_clock": end_clock,
        "duration_seconds": duration,
        "home_lineup": sorted(home_lineup),
        "away_lineup": sorted(away_lineup),
        "score_home_start": score_home_start,
        "score_away_start": score_away_start,
        "score_home_end": score_home_end,
        "score_away_end": score_away_end,
        "ended_by_substitution": bool(ended_by_substitution),
    })

In [11]:
# Build first fact_lineup_stint prototype from grouped timestamps

eg = event_groups_df.copy().sort_values(
    ["period", "clock_seconds_remaining"],
    ascending=[True, False],
    kind="mergesort",
).reset_index(drop=True)

stint_rows = []
cur_period = None
home_lineup = away_lineup = None
cur_start_clock = None
cur_start_sec = None
cur_score_home_start = None
cur_score_away_start = None

for _, row in eg.iterrows():
    period = int(row["period"])
    clock = row["clock"]
    sec = float(row["clock_seconds_remaining"])
    has_sub = bool(row["has_substitution"])

    score_home_here, score_away_here = get_score_snapshot_for_group(pbp_df, period, clock)

    # Start of period: reset lineups and open new stint
    if period != cur_period:
        # close previous period stint at 0:00
        if cur_period is not None:
            append_stint_row(
                stint_rows,
                cur_period,
                cur_start_clock,
                "PT00M00.00S",
                cur_start_sec,
                0.0,
                home_lineup,
                away_lineup,
                cur_score_home_start,
                cur_score_away_start,
                score_home_here,
                score_away_here,
                ended_by_substitution=False,
            )

        cur_period = period
        home_lineup, away_lineup = reset_lineup_state_for_period(period, period_start_lineups)
        cur_start_clock = clock
        cur_start_sec = sec
        cur_score_home_start, cur_score_away_start = score_home_here, score_away_here

    if has_sub:
        # close current stint once at this timestamp
        append_stint_row(
            stint_rows,
            cur_period,
            cur_start_clock,
            clock,
            cur_start_sec,
            sec,
            home_lineup,
            away_lineup,
            cur_score_home_start,
            cur_score_away_start,
            score_home_here,
            score_away_here,
            ended_by_substitution=True,
        )

        # apply entire substitution batch once
        batch = sub_group_lookup.get((period, clock), [])
        home_lineup, away_lineup, _ = apply_substitution_batch_strict(
            home_lineup, away_lineup, batch, home_team_id, away_team_id
        )

        # open one new stint at the same timestamp (no append yet)
        cur_start_clock = clock
        cur_start_sec = sec
        cur_score_home_start, cur_score_away_start = score_home_here, score_away_here

# final flush at game end (end of last period)
if cur_period is not None:
    append_stint_row(
        stint_rows,
        cur_period,
        cur_start_clock,
        "PT00M00.00S",
        cur_start_sec,
        0.0,
        home_lineup,
        away_lineup,
        cur_score_home_start,
        cur_score_away_start,
        cur_score_home_start,
        cur_score_away_start,
        ended_by_substitution=False,
    )

fact_lineup_stint = pd.DataFrame(stint_rows)
print("fact_lineup_stint rows:", len(fact_lineup_stint))
display(fact_lineup_stint.head(20))


[RESET] P1
 HOME: ['Brown', 'Hauser', 'Queta', 'Tatum', 'White']
 AWAY: ['Bona', 'Edgecombe', 'George', 'Maxey', 'Oubre Jr.']
[WARN] HOME: OUT 'Vučević' not found on floor

[RESET] P2
 HOME: ['Brown', 'Pritchard', 'Scheierman', 'Walsh', 'White']
 AWAY: ['Barlow', 'Drummond', 'Edwards', 'George', 'Grimes']
[WARN] HOME: OUT 'Garza' missing, IN 'White' already on floor
[WARN] HOME: OUT 'Hauser' not found on floor
[WARN] HOME: OUT 'Vučević' not found on floor

[RESET] P3
 HOME: ['Brown', 'Hauser', 'Tatum', 'Vucevic', 'White']
 AWAY: ['Bona', 'Edgecombe', 'George', 'Maxey', 'Oubre Jr.']
[WARN] HOME: OUT 'Queta' missing, IN 'Vucevic' already on floor
[WARN] AWAY: OUT 'Drummond' not found on floor
[WARN] AWAY: IN 'Bona' already on floor
[WARN] HOME: OUT 'Vučević' not found on floor
[WARN] AWAY: IN 'Oubre Jr.' already on floor
[WARN] HOME: OUT 'Garza' not found on floor

[RESET] P4
 HOME: ['Brown', 'Hauser', 'Tatum', 'Vucevic', 'White']
 AWAY: ['Drummond', 'Edgecombe', 'George', 'Maxey', 'Oub

,period,start_clock,end_clock,duration_seconds,home_lineup,away_lineup,score_home_start,score_away_start,score_home_end,score_away_end,ended_by_substitution
0,1,PT12M00.00S,PT10M23.00S,97.0,"[Brown, Hauser, Queta, Tatum, White]","[Bona, Edgecombe, George, Maxey, Oubre Jr.]",0.0,0.0,NaN,NaN,True
1,1,PT10M23.00S,PT08M39.00S,104.0,"[Brown, Hauser, Queta, Tatum, White]","[Drummond, Edgecombe, George, Maxey, Oubre Jr.]",NaN,NaN,8.0,7.0,True
2,1,PT08M39.00S,PT04M50.00S,229.0,"[Brown, Hauser, Tatum, Vucevic, White]","[Drummond, Edgecombe, George, Maxey, Oubre Jr.]",8.0,7.0,NaN,NaN,True
3,1,PT04M50.00S,PT04M39.00S,11.0,"[Brown, Pritchard, Tatum, Vucevic, White]","[Barlow, Drummond, George, Grimes, Maxey]",NaN,NaN,22.0,11.0,True
4,1,PT04M39.00S,PT03M49.00S,50.0,"[Brown, Pritchard, Tatum, Vucevic, White]","[Barlow, George, Grimes, Maxey, Oubre Jr.]",22.0,11.0,22.0,14.0,True
5,1,PT03M49.00S,PT02M39.00S,70.0,"[Pritchard, Tatum, Vucevic, Walsh, White]","[Barlow, George, Grimes, Maxey, Oubre Jr.]",22.0,14.0,24.0,16.0,True
6,1,PT02M39.00S,PT00M27.30S,131.7,"[Pritchard, Tatum, Vucevic, Walsh, White]","[Barlow, Edwards, Grimes, Maxey, Oubre Jr.]",24.0,16.0,NaN,NaN,True
7,1,PT00M27.30S,PT00M00.00S,27.3,"[Pritchard, Tatum, Vucevic, Walsh, White]","[Edwards, George, Grimes, Maxey, Oubre Jr.]",NaN,NaN,33.0,18.0,False
8,2,PT12M00.00S,PT08M43.00S,197.0,"[Brown, Pritchard, Scheierman, Walsh, White]","[Barlow, Drummond, Edwards, George, Grimes]",33.0,18.0,42.0,27.0,True
9,2,PT08M43.00S,PT07M16.00S,87.0,"[Brown, Pritchard, Queta, Walsh, White]","[Barlow, Drummond, Edwards, George, Maxey]",42.0,27.0,49.0,29.0,True


In [12]:
# Quick QA on prototype stint table
if len(fact_lineup_stint):
    print("Zero-duration stints:", int((fact_lineup_stint["duration_seconds"] == 0).sum()))
    print("Negative durations:", int((fact_lineup_stint["duration_seconds"] < 0).sum()))
    print("Ended by substitution:", int(fact_lineup_stint["ended_by_substitution"].sum()))
    print("Total seconds:", round(float(fact_lineup_stint["duration_seconds"].sum()), 2))

    display(
        fact_lineup_stint[
            [
                "period", "start_clock", "end_clock", "duration_seconds",
                "home_lineup", "away_lineup", "ended_by_substitution"
            ]
        ].head(20)
    )

Zero-duration stints: 0
Negative durations: 0
Ended by substitution: 31
Total seconds: 2880.0


,period,start_clock,end_clock,duration_seconds,home_lineup,away_lineup,ended_by_substitution
0,1,PT12M00.00S,PT10M23.00S,97.0,"[Brown, Hauser, Queta, Tatum, White]","[Bona, Edgecombe, George, Maxey, Oubre Jr.]",True
1,1,PT10M23.00S,PT08M39.00S,104.0,"[Brown, Hauser, Queta, Tatum, White]","[Drummond, Edgecombe, George, Maxey, Oubre Jr.]",True
2,1,PT08M39.00S,PT04M50.00S,229.0,"[Brown, Hauser, Tatum, Vucevic, White]","[Drummond, Edgecombe, George, Maxey, Oubre Jr.]",True
3,1,PT04M50.00S,PT04M39.00S,11.0,"[Brown, Pritchard, Tatum, Vucevic, White]","[Barlow, Drummond, George, Grimes, Maxey]",True
4,1,PT04M39.00S,PT03M49.00S,50.0,"[Brown, Pritchard, Tatum, Vucevic, White]","[Barlow, George, Grimes, Maxey, Oubre Jr.]",True
5,1,PT03M49.00S,PT02M39.00S,70.0,"[Pritchard, Tatum, Vucevic, Walsh, White]","[Barlow, George, Grimes, Maxey, Oubre Jr.]",True
6,1,PT02M39.00S,PT00M27.30S,131.7,"[Pritchard, Tatum, Vucevic, Walsh, White]","[Barlow, Edwards, Grimes, Maxey, Oubre Jr.]",True
7,1,PT00M27.30S,PT00M00.00S,27.3,"[Pritchard, Tatum, Vucevic, Walsh, White]","[Edwards, George, Grimes, Maxey, Oubre Jr.]",False
8,2,PT12M00.00S,PT08M43.00S,197.0,"[Brown, Pritchard, Scheierman, Walsh, White]","[Barlow, Drummond, Edwards, George, Grimes]",True
9,2,PT08M43.00S,PT07M16.00S,87.0,"[Brown, Pritchard, Queta, Walsh, White]","[Barlow, Drummond, Edwards, George, Maxey]",True


## V2 Refactor (ID-based lineup state)

These cells supersede earlier name-only replay cells.

Design:
- player IDs are source of truth for lineup state
- Q1 starters come from `nba_api.live.nba.endpoints.boxscore` (`starter == 1`)
- Q2/Q3/Q4 start from previous quarter ending lineup, then apply substitutions at `PT12M00.00S`
- names are used only for debug output

In [13]:
# V2 Cell A: Load live boxscore player master + canonical name mappings

import re
import unicodedata
from nba_api.live.nba.endpoints import boxscore


def canon_name(name: str) -> str:
    if name is None:
        return ""
    s = str(name).strip().lower()
    s = unicodedata.normalize("NFKD", s)
    s = "".join(ch for ch in s if not unicodedata.combining(ch))
    s = re.sub(r"[^\w\s]", "", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s


live_bs = boxscore.BoxScore(game_id=GAME_ID).get_dict()["game"]
home_team_id = int(live_bs["homeTeam"]["teamId"])
away_team_id = int(live_bs["awayTeam"]["teamId"])

player_rows = []
for side in ["homeTeam", "awayTeam"]:
    team_id = int(live_bs[side]["teamId"])
    team_tricode = live_bs[side]["teamTricode"]
    for p in live_bs[side]["players"]:
        display_name = p.get("familyName") or p.get("name") or p.get("nameI") or ""
        player_rows.append({
            "player_id": int(p["personId"]),
            "display_name": display_name,
            "normalized_name": canon_name(display_name),
            "team_id": team_id,
            "team_tricode": team_tricode,
            "starter": int(p.get("starter", 0)) == 1,
        })

player_master_df = pd.DataFrame(player_rows).drop_duplicates(subset=["team_id", "player_id"])

# Required helper dicts
player_id_to_name = dict(zip(player_master_df["player_id"], player_master_df["display_name"]))
# team-scoped normalized map is safer than global for collisions
normalized_name_to_player_id = {
    (int(r["team_id"]), r["normalized_name"]): int(r["player_id"])
    for _, r in player_master_df.iterrows()
}

# Q1 starters from live boxscore starter flag
q1_home_starters = set(player_master_df[(player_master_df["team_id"] == home_team_id) & (player_master_df["starter"])]["player_id"].tolist())
q1_away_starters = set(player_master_df[(player_master_df["team_id"] == away_team_id) & (player_master_df["starter"])]["player_id"].tolist())

assert len(q1_home_starters) == 5, f"Expected 5 Q1 home starters, got {len(q1_home_starters)}"
assert len(q1_away_starters) == 5, f"Expected 5 Q1 away starters, got {len(q1_away_starters)}"

print("Teams:", home_team_id, away_team_id)
print("Q1 HOME starters:", sorted(player_id_to_name[i] for i in q1_home_starters))
print("Q1 AWAY starters:", sorted(player_id_to_name[i] for i in q1_away_starters))

display(player_master_df.head(15))

Teams: 1610612738 1610612755
Q1 HOME starters: ['Brown', 'Hauser', 'Queta', 'Tatum', 'White']
Q1 AWAY starters: ['Bona', 'Edgecombe', 'George', 'Maxey', 'Oubre Jr.']


,player_id,display_name,normalized_name,team_id,team_tricode,starter
0,1630573,Hauser,hauser,1610612738,BOS,True
1,1628369,Tatum,tatum,1610612738,BOS,True
2,1629674,Queta,queta,1610612738,BOS,True
3,1627759,Brown,brown,1610612738,BOS,True
4,1628401,White,white,1610612738,BOS,True
5,202696,Vučević,vucevic,1610612738,BOS,False
6,1630202,Pritchard,pritchard,1610612738,BOS,False
7,1641775,Walsh,walsh,1610612738,BOS,False
8,1630568,Garza,garza,1610612738,BOS,False
9,1631248,Scheierman,scheierman,1610612738,BOS,False


In [14]:
# V2 Cell B: Rebuild substitution groups with player IDs

SUB_PATTERN = re.compile(r"^SUB:\s*(?P<player_in>.+?)\s+FOR\s+(?P<player_out>.+?)\s*$", re.IGNORECASE)

sub_df_v2 = pbp_df[pbp_df["actionType"].eq("Substitution")].copy()
parsed = sub_df_v2["description"].astype(str).str.extract(SUB_PATTERN)

sub_df_v2["player_in_name"] = parsed["player_in"].fillna("").str.strip()
sub_df_v2["player_out_name_text"] = parsed["player_out"].fillna("").str.strip()
sub_df_v2["player_out_name_api"] = sub_df_v2["playerName"].fillna("").astype(str).str.strip()

sub_df_v2["team_id"] = sub_df_v2["teamId"].astype(int)
sub_df_v2["team_tricode"] = sub_df_v2["teamTricode"]
sub_df_v2["player_out_id"] = pd.to_numeric(sub_df_v2["personId"], errors="coerce").astype("Int64")

# incoming id via team-scoped canonical lookup
sub_df_v2["player_in_id"] = sub_df_v2.apply(
    lambda r: normalized_name_to_player_id.get((int(r["team_id"]), canon_name(r["player_in_name"]))),
    axis=1,
)

sub_event_cols_v2 = [
    "actionNumber", "actionId", "period", "clock",
    "team_id", "team_tricode",
    "player_out_id", "player_in_id",
    "player_out_name_api", "player_out_name_text", "player_in_name",
    "description",
]

sub_events_clean_v2 = (
    sub_df_v2[sub_event_cols_v2]
    .sort_values(["period", "clock", "actionNumber"], ascending=[True, False, True])
    .reset_index(drop=True)
)

sub_groups_df = (
    sub_events_clean_v2.groupby(["period", "clock"], sort=False)
    .apply(lambda g: pd.Series({"n_subs": len(g), "substitutions": g.to_dict("records")}))
    .reset_index()
)

sub_group_lookup = {
    (row["period"], row["clock"]): row["substitutions"]
    for _, row in sub_groups_df.iterrows()
}

print("Substitution rows:", len(sub_df_v2))
print("Sub batches:", len(sub_groups_df))
print("Incoming ID unresolved:", int(sub_df_v2["player_in_id"].isna().sum()))
display(sub_df_v2[["period", "clock", "team_id", "player_in_name", "player_in_id", "player_out_name_api", "player_out_id"]].head(20))

Substitution rows: 45
Sub batches: 31
Incoming ID unresolved: 0


,period,clock,team_id,player_in_name,player_in_id,player_out_name_api,player_out_id
16,1,PT10M23.00S,1610612755,Drummond,203083,Bona,1641737
34,1,PT08M39.00S,1610612738,Vucevic,202696,Queta,1629674
67,1,PT04M50.00S,1610612738,Pritchard,1630202,Hauser,1630573
68,1,PT04M50.00S,1610612755,Grimes,1629656,Oubre Jr.,1626162
69,1,PT04M50.00S,1610612755,Barlow,1631230,Edgecombe,1642845
74,1,PT04M39.00S,1610612755,Oubre Jr.,1626162,Drummond,203083
85,1,PT03M49.00S,1610612738,Walsh,1641775,Brown,1627759
96,1,PT02M39.00S,1610612755,Edwards,1642348,George,202331
114,1,PT00M27.30S,1610612738,Brown,1627759,Vučević,202696
115,1,PT00M27.30S,1610612755,George,202331,Barlow,1631230


In [15]:
# V2 Cell C: ID-based strict replay with period-start transition logic
# Q2/Q3/Q4 lineup = previous period ending lineup + substitutions at PT12M00.00S

# Keep grouped event logic unchanged, only refresh sub keys
sub_keys = set(zip(sub_groups_df["period"], sub_groups_df["clock"]))
event_groups_df["has_substitution"] = event_groups_df.apply(
    lambda row: (row["period"], row["clock"]) in sub_keys,
    axis=1,
)


def ids_to_display(id_set):
    return sorted(player_id_to_name.get(int(i), f"id:{i}") for i in id_set)


def apply_substitution_batch_strict_ids(
    home_ids: set[int],
    away_ids: set[int],
    sub_batch: list[dict],
    home_team_id: int,
    away_team_id: int,
):
    home = set(home_ids)
    away = set(away_ids)
    debug_rows = []

    for sub in sub_batch:
        team_id = int(sub["team_id"])
        out_id = int(sub["player_out_id"]) if pd.notna(sub["player_out_id"]) else None
        in_id = int(sub["player_in_id"]) if pd.notna(sub["player_in_id"]) else None

        if team_id == home_team_id:
            team_label = "HOME"
            lineup = home
        elif team_id == away_team_id:
            team_label = "AWAY"
            lineup = away
        else:
            debug_rows.append({
                "team": "UNKNOWN",
                "team_id": team_id,
                "player_out_id": out_id,
                "player_in_id": in_id,
                "sub_succeeded": False,
                "error_type": "unknown_team",
            })
            continue

        outgoing_found = (out_id in lineup) if out_id is not None else False
        incoming_already_present = (in_id in lineup) if in_id is not None else False
        incoming_mapped = in_id is not None

        sub_succeeded = incoming_mapped and outgoing_found and (not incoming_already_present)

        if sub_succeeded:
            lineup.remove(out_id)
            lineup.add(in_id)
            error_type = None
            msg = "ok"
        else:
            if not incoming_mapped:
                error_type = "incoming_unmapped"
                msg = f"{team_label}: incoming name unmapped ({sub.get('player_in_name')})"
            elif not outgoing_found and incoming_already_present:
                error_type = "out_missing_and_in_already_present"
                msg = f"{team_label}: OUT missing and IN already present"
            elif not outgoing_found:
                error_type = "outgoing_not_found"
                msg = f"{team_label}: OUT not found"
            else:
                error_type = "incoming_already_present"
                msg = f"{team_label}: IN already present"
            print(f"[WARN] {msg} @ P{sub.get('period')} {sub.get('clock')} {sub.get('description')}")

        debug_rows.append({
            "team": team_label,
            "team_id": team_id,
            "player_out_id": out_id,
            "player_in_id": in_id,
            "player_out": player_id_to_name.get(out_id, sub.get("player_out_name_api")),
            "player_in": player_id_to_name.get(in_id, sub.get("player_in_name")),
            "outgoing_found": outgoing_found,
            "incoming_already_present": incoming_already_present,
            "incoming_mapped": incoming_mapped,
            "sub_succeeded": sub_succeeded,
            "error_type": error_type,
            "description": sub.get("description"),
            "actionNumber": sub.get("actionNumber"),
        })

    return home, away, debug_rows


eg = event_groups_df.copy().sort_values(
    ["period", "clock_seconds_remaining"],
    ascending=[True, False],
    kind="mergesort",
).reset_index(drop=True)

current_period = None
home_state_ids, away_state_ids = None, None
validation_rows_v2 = []

for _, row in eg.iterrows():
    period = int(row["period"])
    clock = row["clock"]
    has_sub = bool(row["has_substitution"])

    if period != current_period:
        # period transition
        if current_period is None:
            # Q1 seeds from live boxscore starters
            home_state_ids = set(q1_home_starters)
            away_state_ids = set(q1_away_starters)
            print("\n[RESET] P1 from live boxscore starters")
            print(" HOME:", ids_to_display(home_state_ids))
            print(" AWAY:", ids_to_display(away_state_ids))
        else:
            # carry previous period ending lineup
            print(f"\n[RESET] P{period} carry previous period ending lineup")
            print(" HOME before 12:00 subs:", ids_to_display(home_state_ids))
            print(" AWAY before 12:00 subs:", ids_to_display(away_state_ids))

            # apply period-start substitutions at PT12M00.00S for this new period
            start_batch = sub_group_lookup.get((period, "PT12M00.00S"), [])
            if start_batch:
                home_state_ids, away_state_ids, _ = apply_substitution_batch_strict_ids(
                    home_state_ids, away_state_ids, start_batch, home_team_id, away_team_id
                )
                print(" HOME after 12:00 subs:", ids_to_display(home_state_ids))
                print(" AWAY after 12:00 subs:", ids_to_display(away_state_ids))

        current_period = period

    if not has_sub:
        continue

    # prevent double-applying the period start batch
    if period > 1 and clock == "PT12M00.00S":
        continue

    batch = sub_group_lookup.get((period, clock), [])
    home_before_ids = sorted(home_state_ids)
    away_before_ids = sorted(away_state_ids)

    home_state_ids, away_state_ids, sub_debug = apply_substitution_batch_strict_ids(
        home_state_ids, away_state_ids, batch, home_team_id, away_team_id
    )

    home_ok = len(home_state_ids) == 5
    away_ok = len(away_state_ids) == 5

    for d in sub_debug:
        validation_rows_v2.append({
            "period": period,
            "clock": clock,
            "team": d["team"],
            "team_id": d["team_id"],
            "player_out": d["player_out"],
            "player_in": d["player_in"],
            "player_out_id": d["player_out_id"],
            "player_in_id": d["player_in_id"],
            "outgoing_found": d["outgoing_found"],
            "incoming_already_present": d["incoming_already_present"],
            "incoming_mapped": d["incoming_mapped"],
            "sub_succeeded": d["sub_succeeded"],
            "error_type": d["error_type"],
            "home_size_ok": home_ok,
            "away_size_ok": away_ok,
            "home_lineup_before": [player_id_to_name.get(i, f"id:{i}") for i in home_before_ids],
            "away_lineup_before": [player_id_to_name.get(i, f"id:{i}") for i in away_before_ids],
            "home_lineup_after": ids_to_display(home_state_ids),
            "away_lineup_after": ids_to_display(away_state_ids),
            "description": d.get("description"),
            "actionNumber": d.get("actionNumber"),
        })

validation_df_v2 = pd.DataFrame(validation_rows_v2)
print("\nReplay V2 complete")
print("Sub rows logged:", len(validation_df_v2))
if len(validation_df_v2):
    print("Sub failures:", int((~validation_df_v2["sub_succeeded"]).sum()))
    print("Home size failures:", int((~validation_df_v2["home_size_ok"]).sum()))
    print("Away size failures:", int((~validation_df_v2["away_size_ok"]).sum()))

display(validation_df_v2.head(30))


[RESET] P1 from live boxscore starters
 HOME: ['Brown', 'Hauser', 'Queta', 'Tatum', 'White']
 AWAY: ['Bona', 'Edgecombe', 'George', 'Maxey', 'Oubre Jr.']

[RESET] P2 carry previous period ending lineup
 HOME before 12:00 subs: ['Brown', 'Pritchard', 'Tatum', 'Walsh', 'White']
 AWAY before 12:00 subs: ['Edwards', 'George', 'Grimes', 'Maxey', 'Oubre Jr.']
[WARN] HOME: OUT missing and IN already present @ PNone None SUB: White FOR Garza
[WARN] HOME: OUT not found @ PNone None SUB: Queta FOR Scheierman
[WARN] AWAY: IN already present @ PNone None SUB: Maxey FOR Grimes
[WARN] HOME: OUT missing and IN already present @ PNone None SUB: Tatum FOR Hauser
[WARN] AWAY: OUT not found @ PNone None SUB: Bona FOR Drummond
[WARN] AWAY: IN already present @ PNone None SUB: Oubre Jr. FOR George
[WARN] AWAY: IN already present @ PNone None SUB: George FOR Edwards
[WARN] HOME: OUT not found @ PNone None SUB: Vucevic FOR Queta
[WARN] HOME: OUT not found @ PNone None SUB: Garza FOR Vucevic
[WARN] AWAY: OUT

,period,clock,team,team_id,player_out,player_in,player_out_id,player_in_id,outgoing_found,incoming_already_present,...,sub_succeeded,error_type,home_size_ok,away_size_ok,home_lineup_before,away_lineup_before,home_lineup_after,away_lineup_after,description,actionNumber
0,1,PT10M23.00S,AWAY,1610612755,Bona,Drummond,1641737,203083,True,False,...,True,NaN,True,True,"[Brown, Tatum, White, Queta, Hauser]","[George, Oubre Jr., Maxey, Bona, Edgecombe]","[Brown, Hauser, Queta, Tatum, White]","[Drummond, Edgecombe, George, Maxey, Oubre Jr.]",SUB: Drummond FOR Bona,24
1,1,PT08M39.00S,HOME,1610612738,Queta,Vučević,1629674,202696,True,False,...,True,NaN,True,True,"[Brown, Tatum, White, Queta, Hauser]","[George, Drummond, Oubre Jr., Maxey, Edgecombe]","[Brown, Hauser, Tatum, Vučević, White]","[Drummond, Edgecombe, George, Maxey, Oubre Jr.]",SUB: Vucevic FOR Queta,49
2,1,PT04M50.00S,HOME,1610612738,Hauser,Pritchard,1630573,1630202,True,False,...,True,NaN,True,True,"[Vučević, Brown, Tatum, White, Hauser]","[George, Drummond, Oubre Jr., Maxey, Edgecombe]","[Brown, Pritchard, Tatum, Vučević, White]","[Barlow, Drummond, George, Grimes, Maxey]",SUB: Pritchard FOR Hauser,91
3,1,PT04M50.00S,AWAY,1610612755,Oubre Jr.,Grimes,1626162,1629656,True,False,...,True,NaN,True,True,"[Vučević, Brown, Tatum, White, Hauser]","[George, Drummond, Oubre Jr., Maxey, Edgecombe]","[Brown, Pritchard, Tatum, Vučević, White]","[Barlow, Drummond, George, Grimes, Maxey]",SUB: Grimes FOR Oubre Jr.,92
4,1,PT04M50.00S,AWAY,1610612755,Edgecombe,Barlow,1642845,1631230,True,False,...,True,NaN,True,True,"[Vučević, Brown, Tatum, White, Hauser]","[George, Drummond, Oubre Jr., Maxey, Edgecombe]","[Brown, Pritchard, Tatum, Vučević, White]","[Barlow, Drummond, George, Grimes, Maxey]",SUB: Barlow FOR Edgecombe,93
5,1,PT04M39.00S,AWAY,1610612755,Drummond,Oubre Jr.,203083,1626162,True,False,...,True,NaN,True,True,"[Vučević, Brown, Tatum, White, Pritchard]","[George, Drummond, Grimes, Maxey, Barlow]","[Brown, Pritchard, Tatum, Vučević, White]","[Barlow, George, Grimes, Maxey, Oubre Jr.]",SUB: Oubre Jr. FOR Drummond,106
6,1,PT03M49.00S,HOME,1610612738,Brown,Walsh,1627759,1641775,True,False,...,True,NaN,True,True,"[Vučević, Brown, Tatum, White, Pritchard]","[George, Oubre Jr., Grimes, Maxey, Barlow]","[Pritchard, Tatum, Vučević, Walsh, White]","[Barlow, George, Grimes, Maxey, Oubre Jr.]",SUB: Walsh FOR Brown,120
7,1,PT02M39.00S,AWAY,1610612755,George,Edwards,202331,1642348,True,False,...,True,NaN,True,True,"[Vučević, Tatum, White, Pritchard, Walsh]","[George, Oubre Jr., Grimes, Maxey, Barlow]","[Pritchard, Tatum, Vučević, Walsh, White]","[Barlow, Edwards, Grimes, Maxey, Oubre Jr.]",SUB: Edwards FOR George,133
8,1,PT00M27.30S,HOME,1610612738,Vučević,Brown,202696,1627759,True,False,...,True,NaN,True,True,"[Vučević, Tatum, White, Pritchard, Walsh]","[Oubre Jr., Grimes, Maxey, Barlow, Edwards]","[Brown, Pritchard, Tatum, Walsh, White]","[Edwards, George, Grimes, Maxey, Oubre Jr.]",SUB: Brown FOR Vucevic,157
9,1,PT00M27.30S,AWAY,1610612755,Barlow,George,1631230,202331,True,False,...,True,NaN,True,True,"[Vučević, Tatum, White, Pritchard, Walsh]","[Oubre Jr., Grimes, Maxey, Barlow, Edwards]","[Brown, Pritchard, Tatum, Walsh, White]","[Edwards, George, Grimes, Maxey, Oubre Jr.]",SUB: George FOR Barlow,158


In [ ]:
# V2 Cell D: First failing substitution per quarter

if validation_df_v2.empty:
    print("No validation rows.")
else:
    fail_mask = (
        (~validation_df_v2["sub_succeeded"])
        | (~validation_df_v2["home_size_ok"])
        | (~validation_df_v2["away_size_ok"])
    )
    fail_df_v2 = validation_df_v2[fail_mask].copy().sort_values(
        ["period", "clock", "actionNumber"],
        ascending=[True, False, True],
    )

    print("Failing rows:", len(fail_df_v2))
    display(fail_df_v2[[
        "period", "clock", "team", "player_out", "player_in",
        "player_out_id", "player_in_id",
        "outgoing_found", "incoming_already_present", "incoming_mapped",
        "sub_succeeded", "error_type", "home_size_ok", "away_size_ok"
    ]])

    first_fail_per_period_v2 = fail_df_v2.groupby("period", as_index=False).first()
    print("\nFirst failure per period:")
    display(first_fail_per_period_v2[[
        "period", "clock", "team", "player_out", "player_in", "error_type",
        "home_lineup_before", "away_lineup_before"
    ]])

In [16]:
# V2 Cell E: Stint prototype using ID state + period transition rule
# NOTE: per request, final stint score handling is unchanged (TODO later).


def append_stint_row_ids(
    out_rows,
    period,
    start_clock,
    end_clock,
    start_sec,
    end_sec,
    home_ids,
    away_ids,
    score_home_start,
    score_away_start,
    score_home_end,
    score_away_end,
    ended_by_substitution,
):
    duration = max(0.0, float(start_sec) - float(end_sec))
    out_rows.append({
        "period": int(period),
        "start_clock": start_clock,
        "end_clock": end_clock,
        "duration_seconds": duration,
        "home_lineup_ids": sorted(home_ids),
        "away_lineup_ids": sorted(away_ids),
        "home_lineup": ids_to_display(home_ids),
        "away_lineup": ids_to_display(away_ids),
        "score_home_start": score_home_start,
        "score_away_start": score_away_start,
        "score_home_end": score_home_end,
        "score_away_end": score_away_end,
        "ended_by_substitution": bool(ended_by_substitution),
    })


eg = event_groups_df.copy().sort_values(
    ["period", "clock_seconds_remaining"],
    ascending=[True, False],
    kind="mergesort",
).reset_index(drop=True)

stint_rows_v2 = []
cur_period = None
home_ids = away_ids = None
cur_start_clock = None
cur_start_sec = None
cur_score_home_start = None
cur_score_away_start = None

for _, row in eg.iterrows():
    period = int(row["period"])
    clock = row["clock"]
    sec = float(row["clock_seconds_remaining"])
    has_sub = bool(row["has_substitution"])

    score_home_here, score_away_here = get_score_snapshot_for_group(pbp_df, period, clock)

    if period != cur_period:
        if cur_period is not None:
            append_stint_row_ids(
                stint_rows_v2,
                cur_period,
                cur_start_clock,
                "PT00M00.00S",
                cur_start_sec,
                0.0,
                home_ids,
                away_ids,
                cur_score_home_start,
                cur_score_away_start,
                score_home_here,
                score_away_here,
                ended_by_substitution=False,
            )

        # new period lineup start
        if period == 1:
            home_ids = set(q1_home_starters)
            away_ids = set(q1_away_starters)
        # else carry previous ending lineup

        if period > 1:
            start_batch = sub_group_lookup.get((period, "PT12M00.00S"), [])
            if start_batch:
                home_ids, away_ids, _ = apply_substitution_batch_strict_ids(
                    home_ids, away_ids, start_batch, home_team_id, away_team_id
                )

        cur_period = period
        cur_start_clock = clock
        cur_start_sec = sec
        cur_score_home_start, cur_score_away_start = score_home_here, score_away_here

    if has_sub:
        # avoid double-applying period-start 12:00 substitutions
        if period > 1 and clock == "PT12M00.00S":
            continue

        append_stint_row_ids(
            stint_rows_v2,
            cur_period,
            cur_start_clock,
            clock,
            cur_start_sec,
            sec,
            home_ids,
            away_ids,
            cur_score_home_start,
            cur_score_away_start,
            score_home_here,
            score_away_here,
            ended_by_substitution=True,
        )

        batch = sub_group_lookup.get((period, clock), [])
        home_ids, away_ids, _ = apply_substitution_batch_strict_ids(
            home_ids, away_ids, batch, home_team_id, away_team_id
        )

        cur_start_clock = clock
        cur_start_sec = sec
        cur_score_home_start, cur_score_away_start = score_home_here, score_away_here

# Final flush (score behavior intentionally unchanged for now)
if cur_period is not None:
    append_stint_row_ids(
        stint_rows_v2,
        cur_period,
        cur_start_clock,
        "PT00M00.00S",
        cur_start_sec,
        0.0,
        home_ids,
        away_ids,
        cur_score_home_start,
        cur_score_away_start,
        cur_score_home_start,
        cur_score_away_start,
        ended_by_substitution=False,
    )

fact_lineup_stint_v2 = pd.DataFrame(stint_rows_v2)
print("fact_lineup_stint_v2 rows:", len(fact_lineup_stint_v2))
display(fact_lineup_stint_v2.head(20))

[WARN] HOME: OUT missing and IN already present @ PNone None SUB: White FOR Garza
[WARN] HOME: OUT not found @ PNone None SUB: Queta FOR Scheierman
[WARN] AWAY: IN already present @ PNone None SUB: Maxey FOR Grimes
[WARN] HOME: OUT missing and IN already present @ PNone None SUB: Tatum FOR Hauser
[WARN] AWAY: OUT not found @ PNone None SUB: Bona FOR Drummond
[WARN] AWAY: IN already present @ PNone None SUB: Oubre Jr. FOR George
[WARN] AWAY: IN already present @ PNone None SUB: George FOR Edwards
[WARN] HOME: OUT not found @ PNone None SUB: Vucevic FOR Queta
[WARN] HOME: OUT not found @ PNone None SUB: Garza FOR Vucevic
[WARN] AWAY: OUT not found @ PNone None SUB: Drummond FOR Bona
[WARN] HOME: OUT not found @ PNone None SUB: Vucevic FOR Queta
[WARN] HOME: IN already present @ PNone None SUB: Pritchard FOR Hauser
[WARN] AWAY: OUT missing and IN already present @ PNone None SUB: Grimes FOR Drummond
[WARN] HOME: IN already present @ PNone None SUB: Walsh FOR Tatum
[WARN] HOME: OUT not fou

,period,start_clock,end_clock,duration_seconds,home_lineup_ids,away_lineup_ids,home_lineup,away_lineup,score_home_start,score_away_start,score_home_end,score_away_end,ended_by_substitution
0,1,PT12M00.00S,PT10M23.00S,97.0,"[1627759, 1628369, 1628401, 1629674, 1630573]","[202331, 1626162, 1630178, 1641737, 1642845]","[Brown, Hauser, Queta, Tatum, White]","[Bona, Edgecombe, George, Maxey, Oubre Jr.]",0.0,0.0,NaN,NaN,True
1,1,PT10M23.00S,PT08M39.00S,104.0,"[1627759, 1628369, 1628401, 1629674, 1630573]","[202331, 203083, 1626162, 1630178, 1642845]","[Brown, Hauser, Queta, Tatum, White]","[Drummond, Edgecombe, George, Maxey, Oubre Jr.]",NaN,NaN,8.0,7.0,True
2,1,PT08M39.00S,PT04M50.00S,229.0,"[202696, 1627759, 1628369, 1628401, 1630573]","[202331, 203083, 1626162, 1630178, 1642845]","[Brown, Hauser, Tatum, Vučević, White]","[Drummond, Edgecombe, George, Maxey, Oubre Jr.]",8.0,7.0,NaN,NaN,True
3,1,PT04M50.00S,PT04M39.00S,11.0,"[202696, 1627759, 1628369, 1628401, 1630202]","[202331, 203083, 1629656, 1630178, 1631230]","[Brown, Pritchard, Tatum, Vučević, White]","[Barlow, Drummond, George, Grimes, Maxey]",NaN,NaN,22.0,11.0,True
4,1,PT04M39.00S,PT03M49.00S,50.0,"[202696, 1627759, 1628369, 1628401, 1630202]","[202331, 1626162, 1629656, 1630178, 1631230]","[Brown, Pritchard, Tatum, Vučević, White]","[Barlow, George, Grimes, Maxey, Oubre Jr.]",22.0,11.0,22.0,14.0,True
5,1,PT03M49.00S,PT02M39.00S,70.0,"[202696, 1628369, 1628401, 1630202, 1641775]","[202331, 1626162, 1629656, 1630178, 1631230]","[Pritchard, Tatum, Vučević, Walsh, White]","[Barlow, George, Grimes, Maxey, Oubre Jr.]",22.0,14.0,24.0,16.0,True
6,1,PT02M39.00S,PT00M27.30S,131.7,"[202696, 1628369, 1628401, 1630202, 1641775]","[1626162, 1629656, 1630178, 1631230, 1642348]","[Pritchard, Tatum, Vučević, Walsh, White]","[Barlow, Edwards, Grimes, Maxey, Oubre Jr.]",24.0,16.0,NaN,NaN,True
7,1,PT00M27.30S,PT00M00.00S,27.3,"[1627759, 1628369, 1628401, 1630202, 1641775]","[202331, 1626162, 1629656, 1630178, 1642348]","[Brown, Pritchard, Tatum, Walsh, White]","[Edwards, George, Grimes, Maxey, Oubre Jr.]",NaN,NaN,33.0,18.0,False
8,2,PT12M00.00S,PT08M43.00S,197.0,"[1627759, 1628369, 1628401, 1630202, 1641775]","[202331, 1626162, 1629656, 1630178, 1642348]","[Brown, Pritchard, Tatum, Walsh, White]","[Edwards, George, Grimes, Maxey, Oubre Jr.]",33.0,18.0,42.0,27.0,True
9,2,PT08M43.00S,PT07M16.00S,87.0,"[1627759, 1628369, 1628401, 1630202, 1641775]","[202331, 1626162, 1629656, 1630178, 1642348]","[Brown, Pritchard, Tatum, Walsh, White]","[Edwards, George, Grimes, Maxey, Oubre Jr.]",42.0,27.0,49.0,29.0,True


In [18]:
#let's check if substitutions happen at 12:00
# Raw rows at 12:00 for periods 2, 3, 4
qstart_rows = pbp_df[
    (pbp_df["period"].isin([2, 3, 4])) &
    (pbp_df["clock"] == "PT12M00.00S")
].copy()

cols = [
    c for c in [
        "actionNumber", "actionId", "period", "clock",
        "teamId", "teamTricode", "personId", "playerName",
        "actionType", "subType", "description",
        "scoreHome", "scoreAway"
    ] if c in qstart_rows.columns
]

print("Total raw rows at 12:00 in Q2/Q3/Q4:", len(qstart_rows))
display(qstart_rows[cols].sort_values(["period", "actionNumber"]))

Total raw rows at 12:00 in Q2/Q3/Q4: 3


,actionNumber,actionId,period,clock,teamId,teamTricode,personId,playerName,actionType,subType,description,scoreHome,scoreAway
124,180,125,2,PT12M00.00S,0,,0,,period,start,Start of 2nd Period (1:45 PM EST),33,18
266,376,269,3,PT12M00.00S,0,,0,,period,start,Start of 3rd Period (2:36 PM EST),64,46
372,527,373,4,PT12M00.00S,0,,0,,period,start,Start of 4th Period (3:08 PM EST),95,71


In [19]:
qstart_subs = pbp_df[
    (pbp_df["period"].isin([2, 3, 4])) &
    (pbp_df["clock"] == "PT12M00.00S") &
    (pbp_df["actionType"].astype(str).str.lower() == "substitution")
].copy()

print("Total substitution rows at 12:00 in Q2/Q3/Q4:", len(qstart_subs))

cols = [
    c for c in [
        "actionNumber", "period", "clock", "teamTricode",
        "personId", "playerName", "actionType", "description"
    ] if c in qstart_subs.columns
]

display(qstart_subs[cols].sort_values(["period", "actionNumber"]))

Total substitution rows at 12:00 in Q2/Q3/Q4: 0


,actionNumber,period,clock,teamTricode,personId,playerName,actionType,description


In [20]:
qstart_sub_groups = sub_groups_df[
    (sub_groups_df["period"].isin([2, 3, 4])) &
    (sub_groups_df["clock"] == "PT12M00.00S")
].copy()

print("Grouped substitution batches at 12:00 in Q2/Q3/Q4:", len(qstart_sub_groups))
display(qstart_sub_groups)

Grouped substitution batches at 12:00 in Q2/Q3/Q4: 0


,period,clock,n_subs,substitutions


In [21]:
for p in [2, 3, 4]:
    print(f"\n=== First 15 raw rows of period {p} ===")
    display(
        pbp_df[pbp_df["period"] == p][[
            c for c in [
                "actionNumber", "clock", "teamTricode", "personId", "playerName",
                "actionType", "subType", "description", "scoreHome", "scoreAway"
            ] if c in pbp_df.columns
        ]].head(15)
    )


=== First 15 raw rows of period 2 ===


,actionNumber,clock,teamTricode,personId,playerName,actionType,subType,description,scoreHome,scoreAway
124,180,PT12M00.00S,,0,,period,start,Start of 2nd Period (1:45 PM EST),33,18
125,181,PT11M39.00S,BOS,1630202,Pritchard,Missed Shot,Pullup Jump shot,MISS Pritchard 29' 3PT Pullup Jump Shot,,
126,182,PT11M37.00S,BOS,1630568,Garza,Rebound,Unknown,Garza REBOUND (Off:1 Def:0),,
127,184,PT11M36.00S,PHI,1642845,Edgecombe,Foul,Shooting,Edgecombe S.FOUL (P1.T1) (P.Fraher),,
128,186,PT11M36.00S,BOS,1630568,Garza,Free Throw,Free Throw 1 of 2,Garza Free Throw 1 of 2 (1 PTS),34,18
129,187,PT11M36.00S,BOS,1630568,Garza,Free Throw,Free Throw 2 of 2,Garza Free Throw 2 of 2 (2 PTS),35,18
130,188,PT11M25.00S,BOS,1630568,Garza,Foul,Shooting,Garza S.FOUL (P1.T1) (P.Fraher),,
131,190,PT11M25.00S,PHI,202331,George,Free Throw,Free Throw 1 of 2,George Free Throw 1 of 2 (6 PTS),35,19
132,191,PT11M25.00S,PHI,202331,George,Free Throw,Free Throw 2 of 2,George Free Throw 2 of 2 (7 PTS),35,20
133,192,PT11M06.00S,BOS,1627759,Brown,Missed Shot,Fadeaway Jump Shot,MISS Brown 12' Fadeaway Jumper,,



=== First 15 raw rows of period 3 ===


,actionNumber,clock,teamTricode,personId,playerName,actionType,subType,description,scoreHome,scoreAway
266,376,PT12M00.00S,,0,,period,start,Start of 3rd Period (2:36 PM EST),64,46
267,377,PT11M45.00S,BOS,1628401,White,Missed Shot,Driving Floating Jump Shot,MISS White 11' Driving Floating Jump Shot,,
268,378,PT11M43.00S,PHI,203083,Drummond,Rebound,Unknown,Drummond REBOUND (Off:0 Def:3),,
269,379,PT11M32.00S,PHI,1630178,Maxey,Missed Shot,Pullup Jump shot,MISS Maxey 16' Pullup Jump Shot,,
270,380,PT11M31.00S,BOS,1628401,White,Rebound,Unknown,White REBOUND (Off:1 Def:2),,
271,381,PT11M17.00S,BOS,1629674,Queta,Made Shot,Cutting Dunk Shot,Queta 1' Cutting Dunk Shot (9 PTS) (White 3 AST),66,46
272,383,PT10M52.00S,PHI,1626162,Oubre Jr.,Made Shot,Driving Finger Roll Layup Shot,Oubre Jr. 3' Driving Finger Roll Layup (6 PTS)...,66,48
273,385,PT10M41.00S,BOS,1628401,White,Made Shot,Pullup Jump shot,White 6' Pullup Jump Shot (7 PTS),68,48
274,386,PT10M41.00S,PHI,1630178,Maxey,Foul,Shooting,Maxey S.FOUL (P3.T1) (P.Fraher),,
275,388,PT10M41.00S,BOS,1628401,White,Free Throw,Free Throw 1 of 1,MISS White Free Throw 1 of 1,,



=== First 15 raw rows of period 4 ===


,actionNumber,clock,teamTricode,personId,playerName,actionType,subType,description,scoreHome,scoreAway
372,527,PT12M00.00S,,0,,period,start,Start of 4th Period (3:08 PM EST),95,71
373,528,PT11M49.00S,PHI,1629656,Grimes,Made Shot,Turnaround Jump Shot,Grimes 18' Turnaround Jump Shot (2 PTS) (Maxey...,95,73
374,530,PT11M38.00S,BOS,1630202,Pritchard,Foul,Offensive,Pritchard OFF.Foul (P2) (S.Foster),,
375,532,PT11M38.00S,BOS,1630202,Pritchard,Turnover,Offensive Foul Turnover,Pritchard Offensive Foul Turnover (P1.T8),,
376,533,PT11M15.00S,PHI,1642845,Edgecombe,Missed Shot,Fadeaway Jump Shot,MISS Edgecombe 10' Fadeaway Jumper,,
377,534,PT11M14.00S,BOS,1629674,Queta,Rebound,Unknown,Queta REBOUND (Off:0 Def:2),,
378,535,PT11M07.00S,BOS,1630573,Hauser,Made Shot,Running Pull-Up Jump Shot,Hauser 31' 3PT Running Pull-Up Jump Shot (9 PT...,98,73
379,537,PT10M46.00S,PHI,1631230,Barlow,Missed Shot,Driving Layup Shot,MISS Barlow 3' Driving Layup,,
380,538,PT10M45.00S,BOS,1631248,Scheierman,Rebound,Unknown,Scheierman REBOUND (Off:0 Def:1),,
381,539,PT10M36.00S,BOS,1628369,Tatum,Missed Shot,Pullup Jump shot,MISS Tatum 25' 3PT Pullup Jump Shot,,


In [22]:
for p in [2, 3, 4]:
    early = pbp_df[
        (pbp_df["period"] == p) &
        (pbp_df["clock_seconds_remaining"] >= 660)   # first minute of quarter
    ].copy()

    players = early[["teamTricode", "personId", "playerName"]].dropna().drop_duplicates()

    print(f"\n=== Players appearing in first minute of Q{p} ===")
    display(players.sort_values(["teamTricode", "playerName"]))


=== Players appearing in first minute of Q2 ===


,teamTricode,personId,playerName
124,,0,
133,BOS,1627759,Brown
126,BOS,1630568,Garza
125,BOS,1630202,Pritchard
127,PHI,1642845,Edgecombe
134,PHI,1642348,Edwards
131,PHI,202331,George



=== Players appearing in first minute of Q3 ===


,teamTricode,personId,playerName
266,,0,
271,BOS,1629674,Queta
267,BOS,1628401,White
268,PHI,203083,Drummond
269,PHI,1630178,Maxey



=== Players appearing in first minute of Q4 ===


,teamTricode,personId,playerName
372,,0,
378,BOS,1630573,Hauser
374,BOS,1630202,Pritchard
377,BOS,1629674,Queta
376,PHI,1642845,Edgecombe
373,PHI,1629656,Grimes


In [24]:
# 1) Q1 game starters from live boxscore
from nba_api.live.nba.endpoints import boxscore as live_boxscore


live_bs = live_boxscore.BoxScore(GAME_ID)
live_dict = live_bs.get_dict()

home_players = live_dict["game"]["homeTeam"]["players"]
away_players = live_dict["game"]["awayTeam"]["players"]

home_q1_starters = [p for p in home_players if p.get("starter")]
away_q1_starters = [p for p in away_players if p.get("starter")]

print("HOME Q1 starters:")
for p in home_q1_starters:
    print(p["personId"], p["name"])

print("\nAWAY Q1 starters:")
for p in away_q1_starters:
    print(p["personId"], p["name"])

HOME Q1 starters:
1630573 Sam Hauser
1628369 Jayson Tatum
1629674 Neemias Queta
1627759 Jaylen Brown
1628401 Derrick White
202696 Nikola Vučević
1630202 Payton Pritchard
1641775 Jordan Walsh
1630568 Luka Garza
1631248 Baylor Scheierman
1631199 Ron Harper Jr.
1642864 Hugo González
1630625 Dalano Banton
1642917 Max Shulga
1642873 Amari Williams

AWAY Q1 starters:
1626162 Kelly Oubre Jr.
202331 Paul George
1641737 Adem Bona
1642845 VJ Edgecombe
1630178 Tyrese Maxey
203083 Andre Drummond
1629656 Quentin Grimes
1631230 Dominick Barlow
1642348 Justin Edwards
1630570 Trendon Watford
1631207 Dalen Terry
1631133 Jabari Walker
1641780 Johni Broome
200768 Kyle Lowry
203954 Joel Embiid


In [26]:
# 2) Quarter-sliced box score: useful as a check, not guaranteed quarter-start lineup truth
from nba_api.stats.endpoints import boxscoretraditionalv3

q2_bs = boxscoretraditionalv3.BoxScoreTraditionalV3(
    game_id=GAME_ID,
    start_period=2,
    end_period=2,
    start_range=0,
    end_range=0,
    range_type=0,
)

q2_df = q2_bs.get_data_frames()[0]   # player_stats
display(q2_df[["teamId", "personId", "firstName", "familyName", "minutes", "points"]].head(20))

,teamId,personId,firstName,familyName,minutes,points
0,1610612738,1630573,Sam,Hauser,28:29,12
1,1610612738,1628369,Jayson,Tatum,32:25,25
2,1610612738,1629674,Neemias,Queta,15:08,13
3,1610612738,1627759,Jaylen,Brown,30:01,26
4,1610612738,1628401,Derrick,White,32:43,10
5,1610612738,202696,Nikola,Vučević,17:30,3
6,1610612738,1630202,Payton,Pritchard,34:24,12
7,1610612738,1641775,Jordan,Walsh,14:40,5
8,1610612738,1630568,Luka,Garza,14:12,7
9,1610612738,1631248,Baylor,Scheierman,15:17,5


In [27]:
# 3) First players who actually appear in Q2/Q3/Q4 play-by-play
for p in [2, 3, 4]:
    early = pbp_df[
        (pbp_df["period"] == p) &
        (pbp_df["clock_seconds_remaining"] >= 660)   # first minute of the quarter
    ].copy()

    players = early[["teamTricode", "personId", "playerName"]].dropna().drop_duplicates()

    print(f"\n=== Players appearing in first minute of Q{p} ===")
    display(players.sort_values(["teamTricode", "playerName"]))


=== Players appearing in first minute of Q2 ===


,teamTricode,personId,playerName
124,,0,
133,BOS,1627759,Brown
126,BOS,1630568,Garza
125,BOS,1630202,Pritchard
127,PHI,1642845,Edgecombe
134,PHI,1642348,Edwards
131,PHI,202331,George



=== Players appearing in first minute of Q3 ===


,teamTricode,personId,playerName
266,,0,
271,BOS,1629674,Queta
267,BOS,1628401,White
268,PHI,203083,Drummond
269,PHI,1630178,Maxey



=== Players appearing in first minute of Q4 ===


,teamTricode,personId,playerName
372,,0,
378,BOS,1630573,Hauser
374,BOS,1630202,Pritchard
377,BOS,1629674,Queta
376,PHI,1642845,Edgecombe
373,PHI,1629656,Grimes


In [28]:
# Cell 1: Helper functions for period-opening lineup inference (ID-first)
# Reasoning:
# - A player seen in a real event before first sub is definitely on floor.
# - A player subbed OUT in first sub batch is definitely on floor immediately before that batch.
# - We infer openers from these hard signals, then fill from previous period end if needed.


def _to_int_or_none(x):
    try:
        if x is None or (isinstance(x, float) and pd.isna(x)):
            return None
        return int(x)
    except Exception:
        return None


def _to_nonempty_str(x):
    if x is None:
        return ""
    s = str(x).strip()
    return "" if s.lower() in {"", "nan", "none"} else s


def _clock_to_seconds(clock_value):
    if pd.isna(clock_value):
        return None
    if "clock_to_seconds_remaining" in globals():
        return clock_to_seconds_remaining(clock_value)
    m = re.match(r"^PT(?P<m>\d+)M(?P<s>\d+(?:\.\d+)?)S$", str(clock_value))
    if not m:
        return None
    return float(m.group("m")) * 60 + float(m.group("s"))


def get_first_substitution_clock(period, sub_groups_df):
    rows = sub_groups_df[sub_groups_df["period"].eq(period)].copy()
    if rows.empty:
        return None
    rows["clock_seconds_remaining"] = rows["clock"].apply(_clock_to_seconds)
    rows = rows.sort_values("clock_seconds_remaining", ascending=False)
    return rows.iloc[0]["clock"]


def get_players_seen_before_first_sub(period, pbp_df, first_sub_clock, home_team_id, away_team_id):
    df = pbp_df[pbp_df["period"].eq(period)].copy()
    if df.empty:
        return {"home": set(), "away": set(), "debug": pd.DataFrame()}

    if "clock_seconds_remaining" not in df.columns:
        df["clock_seconds_remaining"] = df["clock"].apply(_clock_to_seconds)

    start_s = 720.0
    first_sub_s = _clock_to_seconds(first_sub_clock) if first_sub_clock is not None else None
    if first_sub_s is not None:
        df = df[(df["clock_seconds_remaining"] <= start_s) & (df["clock_seconds_remaining"] > first_sub_s)]

    # Keep only real player events. Exclude period-start row, substitutions, blanks, and team events.
    df = df[
        (~df["actionType"].astype(str).str.lower().eq("period"))
        & (~df["actionType"].astype(str).str.lower().eq("substitution"))
    ].copy()

    df["personId_int"] = df["personId"].apply(_to_int_or_none)
    df["teamId_int"] = df["teamId"].apply(_to_int_or_none)
    df["playerName_clean"] = df["playerName"].apply(_to_nonempty_str)

    df = df[(df["personId_int"].notna()) & (df["personId_int"] != 0)]
    df = df[(df["teamId_int"].isin([int(home_team_id), int(away_team_id)]))]
    df = df[df["playerName_clean"] != ""]

    home_seen = set(df[df["teamId_int"].eq(int(home_team_id))]["personId_int"].astype(int).tolist())
    away_seen = set(df[df["teamId_int"].eq(int(away_team_id))]["personId_int"].astype(int).tolist())

    debug_cols = [c for c in ["actionNumber", "clock", "teamId", "teamTricode", "personId", "playerName", "actionType", "description"] if c in df.columns]
    debug = (
        df[debug_cols]
        .drop_duplicates()
        .sort_values([c for c in ["clock", "actionNumber"] if c in df.columns], ascending=[False, True][: len([c for c in ["clock", "actionNumber"] if c in df.columns])])
        .reset_index(drop=True)
    )

    return {"home": home_seen, "away": away_seen, "debug": debug}


def _get_player_in_id(sub):
    in_id = _to_int_or_none(sub.get("player_in_id"))
    if in_id is not None:
        return in_id

    # Optional fallback if your notebook already built normalized_name_to_player_id.
    team_id = _to_int_or_none(sub.get("teamId") or sub.get("team_id"))
    player_in_name = _to_nonempty_str(sub.get("player_in_name") or sub.get("playerInName"))
    if team_id is not None and player_in_name and "normalized_name_to_player_id" in globals() and "canon_name" in globals():
        return _to_int_or_none(normalized_name_to_player_id.get((team_id, canon_name(player_in_name))))

    return None


def get_outgoing_players_from_first_sub(period, first_sub_clock, sub_group_lookup):
    if first_sub_clock is None:
        return {"home": set(), "away": set(), "incoming_home": set(), "incoming_away": set(), "batch": []}

    batch = sub_group_lookup.get((period, first_sub_clock), []) or []
    out_home, out_away = set(), set()
    in_home, in_away = set(), set()

    for sub in batch:
        team_id = _to_int_or_none(sub.get("teamId") or sub.get("team_id"))
        out_id = _to_int_or_none(sub.get("player_out_id"))
        in_id = _get_player_in_id(sub)

        if team_id is None:
            continue
        if out_id is not None:
            if team_id == int(home_team_id):
                out_home.add(out_id)
            elif team_id == int(away_team_id):
                out_away.add(out_id)
        if in_id is not None:
            if team_id == int(home_team_id):
                in_home.add(in_id)
            elif team_id == int(away_team_id):
                in_away.add(in_id)

    return {
        "home": out_home,
        "away": out_away,
        "incoming_home": in_home,
        "incoming_away": in_away,
        "batch": batch,
    }


def infer_opening_lineup_for_period(
    period,
    prev_home_lineup,
    prev_away_lineup,
    pbp_df,
    sub_groups_df,
    sub_group_lookup,
    home_team_id,
    away_team_id,
):
    first_sub_clock = get_first_substitution_clock(period, sub_groups_df)

    seen = get_players_seen_before_first_sub(
        period=period,
        pbp_df=pbp_df,
        first_sub_clock=first_sub_clock,
        home_team_id=home_team_id,
        away_team_id=away_team_id,
    )

    first_sub = get_outgoing_players_from_first_sub(
        period=period,
        first_sub_clock=first_sub_clock,
        sub_group_lookup=sub_group_lookup,
    )

    # Core rule: opener = (players seen before first sub) U (outgoing in first sub batch)
    home_candidates = (seen["home"] | first_sub["home"]) - first_sub["incoming_home"]
    away_candidates = (seen["away"] | first_sub["away"]) - first_sub["incoming_away"]

    home_ambiguous = len(home_candidates) > 5
    away_ambiguous = len(away_candidates) > 5

    # Fill from previous period ending lineup when < 5, excluding incoming first-sub players.
    home_fallback_pool = [pid for pid in prev_home_lineup if pid not in first_sub["incoming_home"]]
    away_fallback_pool = [pid for pid in prev_away_lineup if pid not in first_sub["incoming_away"]]

    inferred_home = set(home_candidates)
    inferred_away = set(away_candidates)

    for pid in home_fallback_pool:
        if len(inferred_home) >= 5:
            break
        inferred_home.add(pid)

    for pid in away_fallback_pool:
        if len(inferred_away) >= 5:
            break
        inferred_away.add(pid)

    # Keep notebook-friendly behavior: if >5 candidates, keep deterministic top-5 but flag ambiguity.
    if len(inferred_home) > 5:
        inferred_home = set(sorted(inferred_home)[:5])
    if len(inferred_away) > 5:
        inferred_away = set(sorted(inferred_away)[:5])

    return {
        "period": period,
        "first_sub_clock": first_sub_clock,
        "seen_before_first_sub_home": seen["home"],
        "seen_before_first_sub_away": seen["away"],
        "outgoing_first_sub_home": first_sub["home"],
        "outgoing_first_sub_away": first_sub["away"],
        "incoming_first_sub_home": first_sub["incoming_home"],
        "incoming_first_sub_away": first_sub["incoming_away"],
        "home_candidates": home_candidates,
        "away_candidates": away_candidates,
        "home_ambiguous": home_ambiguous,
        "away_ambiguous": away_ambiguous,
        "inferred_home": inferred_home,
        "inferred_away": inferred_away,
        "first_sub_batch": first_sub["batch"],
        "pre_sub_debug_events": seen["debug"],
    }


def ids_to_names(ids):
    if "player_id_to_name" in globals():
        return sorted(player_id_to_name.get(int(i), f"id:{i}") for i in ids)
    return sorted([f"id:{int(i)}" for i in ids])

In [29]:
# Cell 2: Q1 starters from live boxscore starter flags
# Reasoning: Q1 is the only period where we trust official starter flags directly.

if "q1_home_starters" not in globals() or "q1_away_starters" not in globals():
    if "player_master_df" in globals():
        q1_home_starters = set(
            player_master_df[
                (player_master_df["team_id"].eq(int(home_team_id))) & (player_master_df["starter"])
            ]["player_id"].astype(int).tolist()
        )
        q1_away_starters = set(
            player_master_df[
                (player_master_df["team_id"].eq(int(away_team_id))) & (player_master_df["starter"])
            ]["player_id"].astype(int).tolist()
        )
    else:
        from nba_api.live.nba.endpoints import boxscore as live_boxscore

        live_bs = live_boxscore.BoxScore(game_id=GAME_ID).get_dict()["game"]
        home_players = live_bs["homeTeam"]["players"]
        away_players = live_bs["awayTeam"]["players"]
        q1_home_starters = {int(p["personId"]) for p in home_players if int(p.get("starter", 0)) == 1}
        q1_away_starters = {int(p["personId"]) for p in away_players if int(p.get("starter", 0)) == 1}

assert len(q1_home_starters) == 5, f"Expected 5 Q1 home starters, got {len(q1_home_starters)}"
assert len(q1_away_starters) == 5, f"Expected 5 Q1 away starters, got {len(q1_away_starters)}"

print("Q1 HOME starters (IDs):", sorted(q1_home_starters))
print("Q1 AWAY starters (IDs):", sorted(q1_away_starters))
print("Q1 HOME starters (names):", ids_to_names(q1_home_starters))
print("Q1 AWAY starters (names):", ids_to_names(q1_away_starters))

Q1 HOME starters (IDs): [1627759, 1628369, 1628401, 1629674, 1630573]
Q1 AWAY starters (IDs): [202331, 1626162, 1630178, 1641737, 1642845]
Q1 HOME starters (names): ['Brown', 'Hauser', 'Queta', 'Tatum', 'White']
Q1 AWAY starters (names): ['Bona', 'Edgecombe', 'George', 'Maxey', 'Oubre Jr.']


In [30]:
# Cell 3: Infer Q2/Q3/Q4 opening lineups from your own data
# Reasoning:
# 1) Use previous period ending lineup as baseline.
# 2) Infer opening lineup from seen-before-first-sub + outgoing first-sub players.
# 3) Iterate period-by-period so each next period uses actual prior end state.


def apply_sub_batch_ids(home_ids, away_ids, sub_batch, home_team_id, away_team_id):
    home = set(home_ids)
    away = set(away_ids)
    debug = []

    for sub in (sub_batch or []):
        team_id = _to_int_or_none(sub.get("teamId") or sub.get("team_id"))
        out_id = _to_int_or_none(sub.get("player_out_id"))
        in_id = _get_player_in_id(sub)

        if team_id == int(home_team_id):
            lineup = home
            team = "HOME"
        elif team_id == int(away_team_id):
            lineup = away
            team = "AWAY"
        else:
            debug.append({"team": "UNKNOWN", "ok": False, "reason": f"unknown teamId {team_id}"})
            continue

        outgoing_found = out_id in lineup if out_id is not None else False
        incoming_already = in_id in lineup if in_id is not None else False
        ok = outgoing_found and (not incoming_already) and (in_id is not None)

        if ok:
            lineup.remove(out_id)
            lineup.add(in_id)

        debug.append(
            {
                "team": team,
                "out_id": out_id,
                "in_id": in_id,
                "outgoing_found": outgoing_found,
                "incoming_already_present": incoming_already,
                "applied": ok,
            }
        )

    return home, away, debug


def get_period_end_lineups_from_openers(period, home_open_ids, away_open_ids, sub_groups_df, sub_group_lookup):
    period_subs = sub_groups_df[sub_groups_df["period"].eq(period)].copy()
    if period_subs.empty:
        return set(home_open_ids), set(away_open_ids)

    period_subs["clock_seconds_remaining"] = period_subs["clock"].apply(_clock_to_seconds)
    period_subs = period_subs.sort_values("clock_seconds_remaining", ascending=False)

    home_state, away_state = set(home_open_ids), set(away_open_ids)
    for _, row in period_subs.iterrows():
        batch = sub_group_lookup.get((int(row["period"]), row["clock"]), [])
        home_state, away_state, _ = apply_sub_batch_ids(
            home_state,
            away_state,
            batch,
            home_team_id=home_team_id,
            away_team_id=away_team_id,
        )

    return home_state, away_state


period_inference = {}

# Seed with official Q1 starters.
period_inference[1] = {
    "inferred_home": set(q1_home_starters),
    "inferred_away": set(q1_away_starters),
    "first_sub_clock": get_first_substitution_clock(1, sub_groups_df),
}

prev_home_end, prev_away_end = get_period_end_lineups_from_openers(
    period=1,
    home_open_ids=q1_home_starters,
    away_open_ids=q1_away_starters,
    sub_groups_df=sub_groups_df,
    sub_group_lookup=sub_group_lookup,
)

for period in [2, 3, 4]:
    inf = infer_opening_lineup_for_period(
        period=period,
        prev_home_lineup=prev_home_end,
        prev_away_lineup=prev_away_end,
        pbp_df=pbp_df,
        sub_groups_df=sub_groups_df,
        sub_group_lookup=sub_group_lookup,
        home_team_id=home_team_id,
        away_team_id=away_team_id,
    )
    period_inference[period] = inf

    print(f"\n===== PERIOD {period} =====")
    print("First sub clock:", inf["first_sub_clock"])

    print("Seen before first sub - HOME IDs:", sorted(inf["seen_before_first_sub_home"]))
    print("Seen before first sub - HOME names:", ids_to_names(inf["seen_before_first_sub_home"]))
    print("Seen before first sub - AWAY IDs:", sorted(inf["seen_before_first_sub_away"]))
    print("Seen before first sub - AWAY names:", ids_to_names(inf["seen_before_first_sub_away"]))

    print("Outgoing first-sub - HOME IDs:", sorted(inf["outgoing_first_sub_home"]))
    print("Outgoing first-sub - HOME names:", ids_to_names(inf["outgoing_first_sub_home"]))
    print("Outgoing first-sub - AWAY IDs:", sorted(inf["outgoing_first_sub_away"]))
    print("Outgoing first-sub - AWAY names:", ids_to_names(inf["outgoing_first_sub_away"]))

    print("Inferred opening HOME IDs:", sorted(inf["inferred_home"]))
    print("Inferred opening HOME names:", ids_to_names(inf["inferred_home"]))
    print("Inferred opening AWAY IDs:", sorted(inf["inferred_away"]))
    print("Inferred opening AWAY names:", ids_to_names(inf["inferred_away"]))

    if inf["home_ambiguous"] or inf["away_ambiguous"]:
        print("[AMBIGUITY FLAG]")
        if inf["home_ambiguous"]:
            print("  HOME candidates (>5):", sorted(inf["home_candidates"]), ids_to_names(inf["home_candidates"]))
        if inf["away_ambiguous"]:
            print("  AWAY candidates (>5):", sorted(inf["away_candidates"]), ids_to_names(inf["away_candidates"]))

    # advance end-of-period state for next iteration
    prev_home_end, prev_away_end = get_period_end_lineups_from_openers(
        period=period,
        home_open_ids=inf["inferred_home"],
        away_open_ids=inf["inferred_away"],
        sub_groups_df=sub_groups_df,
        sub_group_lookup=sub_group_lookup,
    )


===== PERIOD 2 =====
First sub clock: PT08M43.00S
Seen before first sub - HOME IDs: [1627759, 1630202, 1630568, 1630573, 1631248]
Seen before first sub - HOME names: ['Brown', 'Garza', 'Hauser', 'Pritchard', 'Scheierman']
Seen before first sub - AWAY IDs: [202331, 203083, 1629656, 1642348, 1642845]
Seen before first sub - AWAY names: ['Drummond', 'Edgecombe', 'Edwards', 'George', 'Grimes']
Outgoing first-sub - HOME IDs: [1630568, 1631248]
Outgoing first-sub - HOME names: ['Garza', 'Scheierman']
Outgoing first-sub - AWAY IDs: [1629656]
Outgoing first-sub - AWAY names: ['Grimes']
Inferred opening HOME IDs: [1627759, 1630202, 1630568, 1630573, 1631248]
Inferred opening HOME names: ['Brown', 'Garza', 'Hauser', 'Pritchard', 'Scheierman']
Inferred opening AWAY IDs: [202331, 203083, 1629656, 1642348, 1642845]
Inferred opening AWAY names: ['Drummond', 'Edgecombe', 'Edwards', 'George', 'Grimes']

===== PERIOD 3 =====
First sub clock: PT10M25.00S
Seen before first sub - HOME IDs: [1628401, 1629

In [31]:
# Cell 4: Validate inferred opening lineups by applying first-sub batch
# Reasoning: valid opening lineup should satisfy strict sub mechanics on first substitution batch.


def validate_first_sub_application(open_home, open_away, first_sub_batch):
    home = set(open_home)
    away = set(open_away)

    checks = {
        "outgoing_players_found": True,
        "incoming_players_not_already_present": True,
        "lineup_size_remains_5": True,
    }
    per_sub_debug = []

    for sub in (first_sub_batch or []):
        team_id = _to_int_or_none(sub.get("teamId") or sub.get("team_id"))
        out_id = _to_int_or_none(sub.get("player_out_id"))
        in_id = _get_player_in_id(sub)

        if team_id == int(home_team_id):
            lineup = home
            team = "HOME"
        elif team_id == int(away_team_id):
            lineup = away
            team = "AWAY"
        else:
            checks["outgoing_players_found"] = False
            checks["incoming_players_not_already_present"] = False
            per_sub_debug.append({"team": "UNKNOWN", "error": f"unknown teamId {team_id}"})
            continue

        outgoing_found = (out_id is not None) and (out_id in lineup)
        incoming_already = (in_id is not None) and (in_id in lineup)

        if not outgoing_found:
            checks["outgoing_players_found"] = False
        if incoming_already or (in_id is None):
            checks["incoming_players_not_already_present"] = False

        if outgoing_found and (not incoming_already) and (in_id is not None):
            lineup.remove(out_id)
            lineup.add(in_id)

        per_sub_debug.append(
            {
                "team": team,
                "out_id": out_id,
                "in_id": in_id,
                "outgoing_found": outgoing_found,
                "incoming_already_present": incoming_already,
            }
        )

    checks["lineup_size_remains_5"] = (len(home) == 5) and (len(away) == 5)
    checks["validation_pass"] = all(checks.values())
    checks["home_after"] = home
    checks["away_after"] = away
    checks["per_sub_debug"] = per_sub_debug
    return checks


period_first_sub_validation = {}

for period in [2, 3, 4]:
    inf = period_inference[period]
    val = validate_first_sub_application(
        open_home=inf["inferred_home"],
        open_away=inf["inferred_away"],
        first_sub_batch=inf["first_sub_batch"],
    )
    period_first_sub_validation[period] = val

    print(f"\n===== VALIDATION PERIOD {period} =====")
    print("outgoing players found:", val["outgoing_players_found"])
    print("incoming players not already present:", val["incoming_players_not_already_present"])
    print("lineup size remains 5:", val["lineup_size_remains_5"])
    print("VALIDATION PASS:", val["validation_pass"])

    if not val["validation_pass"]:
        print("Per-sub debug:")
        display(pd.DataFrame(val["per_sub_debug"]))


===== VALIDATION PERIOD 2 =====
outgoing players found: True
incoming players not already present: True
lineup size remains 5: True
VALIDATION PASS: True

===== VALIDATION PERIOD 3 =====
outgoing players found: True
incoming players not already present: True
lineup size remains 5: True
VALIDATION PASS: True

===== VALIDATION PERIOD 4 =====
outgoing players found: True
incoming players not already present: True
lineup size remains 5: True
VALIDATION PASS: True


In [32]:
# Cell 5: Final period-start structure (ID-first)
# Reasoning: produce one inspectable object for downstream lineup replay/stint build.

period_start_lineups_inferred = {
    1: {"home": set(q1_home_starters), "away": set(q1_away_starters)},
    2: {"home": set(period_inference[2]["inferred_home"]), "away": set(period_inference[2]["inferred_away"])},
    3: {"home": set(period_inference[3]["inferred_home"]), "away": set(period_inference[3]["inferred_away"])},
    4: {"home": set(period_inference[4]["inferred_home"]), "away": set(period_inference[4]["inferred_away"])},
}

print("period_start_lineups_inferred (IDs):")
for p in [1, 2, 3, 4]:
    print(f"P{p} HOME:", sorted(period_start_lineups_inferred[p]["home"]))
    print(f"P{p} AWAY:", sorted(period_start_lineups_inferred[p]["away"]))

print("\nperiod_start_lineups_inferred (names):")
for p in [1, 2, 3, 4]:
    print(f"P{p} HOME:", ids_to_names(period_start_lineups_inferred[p]["home"]))
    print(f"P{p} AWAY:", ids_to_names(period_start_lineups_inferred[p]["away"]))

period_start_lineups_inferred (IDs):
P1 HOME: [1627759, 1628369, 1628401, 1629674, 1630573]
P1 AWAY: [202331, 1626162, 1630178, 1641737, 1642845]
P2 HOME: [1627759, 1630202, 1630568, 1630573, 1631248]
P2 AWAY: [202331, 203083, 1629656, 1642348, 1642845]
P3 HOME: [1628369, 1628401, 1629674, 1630202, 1630568]
P3 AWAY: [202331, 203083, 1626162, 1630178, 1642845]
P4 HOME: [1628369, 1629674, 1630202, 1630573, 1631248]
P4 AWAY: [1629656, 1630178, 1631230, 1642348, 1642845]

period_start_lineups_inferred (names):
P1 HOME: ['Brown', 'Hauser', 'Queta', 'Tatum', 'White']
P1 AWAY: ['Bona', 'Edgecombe', 'George', 'Maxey', 'Oubre Jr.']
P2 HOME: ['Brown', 'Garza', 'Hauser', 'Pritchard', 'Scheierman']
P2 AWAY: ['Drummond', 'Edgecombe', 'Edwards', 'George', 'Grimes']
P3 HOME: ['Garza', 'Pritchard', 'Queta', 'Tatum', 'White']
P3 AWAY: ['Drummond', 'Edgecombe', 'George', 'Maxey', 'Oubre Jr.']
P4 HOME: ['Hauser', 'Pritchard', 'Queta', 'Scheierman', 'Tatum']
P4 AWAY: ['Barlow', 'Edgecombe', 'Edwards', 'Gr

Testing new logic

In [33]:
# Cell 1 (v2): First substitution role per player in quarter
# Rule: player's first substitution role in quarter tells opening-floor status.
# - first role OUT => started quarter
# - first role IN  => did not start quarter


def _extract_sub_in_out_ids(sub_row):
    out_id = _to_int_or_none(sub_row.get("player_out_id"))
    in_id = _get_player_in_id(sub_row)
    team_id = _to_int_or_none(sub_row.get("teamId") or sub_row.get("team_id"))
    return team_id, out_id, in_id


def get_first_sub_roles_for_period(period, sub_groups_df, home_team_id, away_team_id):
    # Flatten quarter sub rows in true game order (start -> end of period).
    q_subs = sub_groups_df[sub_groups_df["period"].eq(period)].copy()
    if q_subs.empty:
        return {
            "home": {},
            "away": {},
            "home_out": set(),
            "away_out": set(),
            "home_in": set(),
            "away_in": set(),
            "ordered_sub_rows": [],
        }

    q_subs["clock_seconds_remaining"] = q_subs["clock"].apply(_clock_to_seconds)
    q_subs = q_subs.sort_values("clock_seconds_remaining", ascending=False)

    ordered_rows = []
    for _, g in q_subs.iterrows():
        subs = g["substitutions"] or []
        # Preserve within-batch order by actionNumber when present.
        subs_sorted = sorted(subs, key=lambda r: _to_int_or_none(r.get("actionNumber")) or 10**9)
        ordered_rows.extend(subs_sorted)

    home_roles = {}
    away_roles = {}

    for sub in ordered_rows:
        team_id, out_id, in_id = _extract_sub_in_out_ids(sub)
        if team_id not in {int(home_team_id), int(away_team_id)}:
            continue

        role_map = home_roles if team_id == int(home_team_id) else away_roles

        if out_id is not None and out_id not in role_map:
            role_map[out_id] = "OUT"
        if in_id is not None and in_id not in role_map:
            role_map[in_id] = "IN"

    home_out = {pid for pid, role in home_roles.items() if role == "OUT"}
    away_out = {pid for pid, role in away_roles.items() if role == "OUT"}
    home_in = {pid for pid, role in home_roles.items() if role == "IN"}
    away_in = {pid for pid, role in away_roles.items() if role == "IN"}

    return {
        "home": home_roles,
        "away": away_roles,
        "home_out": home_out,
        "away_out": away_out,
        "home_in": home_in,
        "away_in": away_in,
        "ordered_sub_rows": ordered_rows,
    }

In [34]:
# Cell 2 (v2): Improved opening-lineup inference using first-sub-role evidence
# Priority evidence per team:
#   1) first-role OUT locks
#   2) early-quarter observed real-event players
#   3) fallback prior (Q3 -> Q1 starters, else previous quarter end)
# Exclusion rule:
#   - never add players whose first role in quarter is IN


def _ordered_unique(source_ids):
    out = []
    seen = set()
    for x in source_ids:
        i = _to_int_or_none(x)
        if i is None or i in seen:
            continue
        seen.add(i)
        out.append(i)
    return out


def _infer_team_opening_lineup_v2(
    team_label,
    out_locks,
    in_exclusions,
    observed_early,
    fallback_prior,
):
    # Build in deterministic order for notebook inspection.
    candidate_order = []

    # 1) Strongest: first-role OUT locks.
    candidate_order.extend(sorted(out_locks))

    # 2) Strong signal: early-quarter observed real-event players.
    # Only add if not explicitly first-role IN.
    candidate_order.extend([pid for pid in sorted(observed_early) if pid not in in_exclusions])

    # 3) Prior fill.
    candidate_order.extend([pid for pid in _ordered_unique(fallback_prior) if pid not in in_exclusions])

    inferred = []
    for pid in candidate_order:
        if pid in in_exclusions:
            continue
        if pid not in inferred:
            inferred.append(pid)
        if len(inferred) == 5:
            break

    details = {
        "team": team_label,
        "out_locks": set(out_locks),
        "in_exclusions": set(in_exclusions),
        "observed_early": set(observed_early),
        "fallback_prior": list(_ordered_unique(fallback_prior)),
        "candidate_order": candidate_order,
        "shortfall": max(0, 5 - len(inferred)),
    }

    # If still short, keep as-is and flag; caller can inspect.
    return set(inferred), details


def infer_opening_lineup_for_period_v2(
    period,
    prev_home_lineup,
    prev_away_lineup,
    pbp_df,
    sub_groups_df,
    sub_group_lookup,
    home_team_id,
    away_team_id,
    q1_home_starters,
    q1_away_starters,
):
    first_sub_clock = get_first_substitution_clock(period, sub_groups_df)
    seen = get_players_seen_before_first_sub(
        period=period,
        pbp_df=pbp_df,
        first_sub_clock=first_sub_clock,
        home_team_id=home_team_id,
        away_team_id=away_team_id,
    )

    roles = get_first_sub_roles_for_period(
        period=period,
        sub_groups_df=sub_groups_df,
        home_team_id=home_team_id,
        away_team_id=away_team_id,
    )

    # Rule 3: fallback prior
    if period == 3:
        fallback_home = list(sorted(q1_home_starters))
        fallback_away = list(sorted(q1_away_starters))
        fallback_source = "Q1 starters"
    else:
        fallback_home = list(sorted(prev_home_lineup))
        fallback_away = list(sorted(prev_away_lineup))
        fallback_source = "previous quarter ending lineup"

    inferred_home, home_debug = _infer_team_opening_lineup_v2(
        team_label="HOME",
        out_locks=roles["home_out"],
        in_exclusions=roles["home_in"],
        observed_early=seen["home"],
        fallback_prior=fallback_home,
    )
    inferred_away, away_debug = _infer_team_opening_lineup_v2(
        team_label="AWAY",
        out_locks=roles["away_out"],
        in_exclusions=roles["away_in"],
        observed_early=seen["away"],
        fallback_prior=fallback_away,
    )

    return {
        "period": period,
        "first_sub_clock": first_sub_clock,
        "fallback_source": fallback_source,
        "first_sub_roles": roles,
        "seen_before_first_sub_home": seen["home"],
        "seen_before_first_sub_away": seen["away"],
        "inferred_home": inferred_home,
        "inferred_away": inferred_away,
        "home_debug": home_debug,
        "away_debug": away_debug,
        "first_sub_batch": sub_group_lookup.get((period, first_sub_clock), []) if first_sub_clock else [],
    }

In [35]:
# Cell 3 (v2): Debugging output for each period/team under new inference rules

period_inference_v2 = {
    1: {
        "period": 1,
        "inferred_home": set(q1_home_starters),
        "inferred_away": set(q1_away_starters),
        "fallback_source": "official Q1 starters",
    }
}

# End-of-Q1 state is still needed as prior for Q2.
prev_home_end_v2, prev_away_end_v2 = get_period_end_lineups_from_openers(
    period=1,
    home_open_ids=q1_home_starters,
    away_open_ids=q1_away_starters,
    sub_groups_df=sub_groups_df,
    sub_group_lookup=sub_group_lookup,
)

for period in [2, 3, 4]:
    inf = infer_opening_lineup_for_period_v2(
        period=period,
        prev_home_lineup=prev_home_end_v2,
        prev_away_lineup=prev_away_end_v2,
        pbp_df=pbp_df,
        sub_groups_df=sub_groups_df,
        sub_group_lookup=sub_group_lookup,
        home_team_id=home_team_id,
        away_team_id=away_team_id,
        q1_home_starters=q1_home_starters,
        q1_away_starters=q1_away_starters,
    )
    period_inference_v2[period] = inf

    print(f"\n================ PERIOD {period} (v2) ================")
    print("First sub clock:", inf["first_sub_clock"])
    print("Fallback prior used:", inf["fallback_source"])

    roles = inf["first_sub_roles"]

    print("\nHOME first-sub-role OUT:", sorted(roles["home_out"]), ids_to_names(roles["home_out"]))
    print("HOME first-sub-role IN :", sorted(roles["home_in"]), ids_to_names(roles["home_in"]))
    print("HOME early observed    :", sorted(inf["seen_before_first_sub_home"]), ids_to_names(inf["seen_before_first_sub_home"]))
    print("HOME fallback prior    :", inf["home_debug"]["fallback_prior"], ids_to_names(inf["home_debug"]["fallback_prior"]))
    print("HOME inferred lineup   :", sorted(inf["inferred_home"]), ids_to_names(inf["inferred_home"]))
    if len(inf["inferred_home"]) != 5:
        print("[WARN] HOME inferred lineup size != 5")

    print("\nAWAY first-sub-role OUT:", sorted(roles["away_out"]), ids_to_names(roles["away_out"]))
    print("AWAY first-sub-role IN :", sorted(roles["away_in"]), ids_to_names(roles["away_in"]))
    print("AWAY early observed    :", sorted(inf["seen_before_first_sub_away"]), ids_to_names(inf["seen_before_first_sub_away"]))
    print("AWAY fallback prior    :", inf["away_debug"]["fallback_prior"], ids_to_names(inf["away_debug"]["fallback_prior"]))
    print("AWAY inferred lineup   :", sorted(inf["inferred_away"]), ids_to_names(inf["inferred_away"]))
    if len(inf["inferred_away"]) != 5:
        print("[WARN] AWAY inferred lineup size != 5")

    # advance previous quarter ending state using inferred openers
    prev_home_end_v2, prev_away_end_v2 = get_period_end_lineups_from_openers(
        period=period,
        home_open_ids=inf["inferred_home"],
        away_open_ids=inf["inferred_away"],
        sub_groups_df=sub_groups_df,
        sub_group_lookup=sub_group_lookup,
    )


================ PERIOD 2 (v2) ================
First sub clock: PT08M43.00S
Fallback prior used: previous quarter ending lineup

HOME first-sub-role OUT: [1627759, 1630202, 1630568, 1630573, 1631248] ['Brown', 'Garza', 'Hauser', 'Pritchard', 'Scheierman']
HOME first-sub-role IN : [202696, 1628369, 1628401, 1629674] ['Queta', 'Tatum', 'Vučević', 'White']
HOME early observed    : [1627759, 1630202, 1630568, 1630573, 1631248] ['Brown', 'Garza', 'Hauser', 'Pritchard', 'Scheierman']
HOME fallback prior    : [1627759, 1628369, 1628401, 1630202, 1641775] ['Brown', 'Pritchard', 'Tatum', 'Walsh', 'White']
HOME inferred lineup   : [1627759, 1630202, 1630568, 1630573, 1631248] ['Brown', 'Garza', 'Hauser', 'Pritchard', 'Scheierman']

AWAY first-sub-role OUT: [202331, 203083, 1629656, 1642348] ['Drummond', 'Edwards', 'George', 'Grimes']
AWAY first-sub-role IN : [1626162, 1630178, 1641737] ['Bona', 'Maxey', 'Oubre Jr.']
AWAY early observed    : [202331, 203083, 1629656, 1642348, 1642845] ['Drummon

In [36]:
# Cell 4 (v2): Rebuild period_start_lineups_inferred from improved inference

period_start_lineups_inferred = {
    1: {"home": set(q1_home_starters), "away": set(q1_away_starters)},
    2: {"home": set(period_inference_v2[2]["inferred_home"]), "away": set(period_inference_v2[2]["inferred_away"])},
    3: {"home": set(period_inference_v2[3]["inferred_home"]), "away": set(period_inference_v2[3]["inferred_away"])},
    4: {"home": set(period_inference_v2[4]["inferred_home"]), "away": set(period_inference_v2[4]["inferred_away"])},
}

print("period_start_lineups_inferred (IDs)")
for p in [1, 2, 3, 4]:
    print(f"P{p} HOME:", sorted(period_start_lineups_inferred[p]["home"]))
    print(f"P{p} AWAY:", sorted(period_start_lineups_inferred[p]["away"]))

print("\nperiod_start_lineups_inferred (names)")
for p in [1, 2, 3, 4]:
    print(f"P{p} HOME:", ids_to_names(period_start_lineups_inferred[p]["home"]))
    print(f"P{p} AWAY:", ids_to_names(period_start_lineups_inferred[p]["away"]))

period_start_lineups_inferred (IDs)
P1 HOME: [1627759, 1628369, 1628401, 1629674, 1630573]
P1 AWAY: [202331, 1626162, 1630178, 1641737, 1642845]
P2 HOME: [1627759, 1630202, 1630568, 1630573, 1631248]
P2 AWAY: [202331, 203083, 1629656, 1642348, 1642845]
P3 HOME: [1627759, 1628369, 1628401, 1629674, 1630573]
P3 AWAY: [202331, 203083, 1626162, 1630178, 1642845]
P4 HOME: [1628369, 1629674, 1630202, 1630573, 1631248]
P4 AWAY: [1629656, 1630178, 1631230, 1642348, 1642845]

period_start_lineups_inferred (names)
P1 HOME: ['Brown', 'Hauser', 'Queta', 'Tatum', 'White']
P1 AWAY: ['Bona', 'Edgecombe', 'George', 'Maxey', 'Oubre Jr.']
P2 HOME: ['Brown', 'Garza', 'Hauser', 'Pritchard', 'Scheierman']
P2 AWAY: ['Drummond', 'Edgecombe', 'Edwards', 'George', 'Grimes']
P3 HOME: ['Brown', 'Hauser', 'Queta', 'Tatum', 'White']
P3 AWAY: ['Drummond', 'Edgecombe', 'George', 'Maxey', 'Oubre Jr.']
P4 HOME: ['Hauser', 'Pritchard', 'Queta', 'Scheierman', 'Tatum']
P4 AWAY: ['Barlow', 'Edgecombe', 'Edwards', 'Grimes'

In [37]:
# Cell 5 (v2): Stronger validation over first N substitution batches in quarter
# Checks:
# - outgoing is present
# - incoming is not already present
# - lineup sizes stay 5


def validate_n_sub_batches(period, home_open_ids, away_open_ids, sub_groups_df, sub_group_lookup, n_batches=3):
    q_subs = sub_groups_df[sub_groups_df["period"].eq(period)].copy()
    if q_subs.empty:
        return {
            "period": period,
            "n_batches_checked": 0,
            "validation_pass": True,
            "rows": [],
            "home_final": set(home_open_ids),
            "away_final": set(away_open_ids),
        }

    q_subs["clock_seconds_remaining"] = q_subs["clock"].apply(_clock_to_seconds)
    q_subs = q_subs.sort_values("clock_seconds_remaining", ascending=False).head(n_batches)

    home = set(home_open_ids)
    away = set(away_open_ids)
    rows = []

    for _, r in q_subs.iterrows():
        clock = r["clock"]
        batch = sub_group_lookup.get((period, clock), []) or []

        for sub in batch:
            team_id, out_id, in_id = _extract_sub_in_out_ids(sub)
            if team_id == int(home_team_id):
                lineup = home
                team = "HOME"
            elif team_id == int(away_team_id):
                lineup = away
                team = "AWAY"
            else:
                rows.append({
                    "period": period,
                    "clock": clock,
                    "team": "UNKNOWN",
                    "out_id": out_id,
                    "in_id": in_id,
                    "outgoing_found": False,
                    "incoming_already_present": False,
                    "size_ok": (len(home) == 5 and len(away) == 5),
                    "applied": False,
                    "error": f"unknown teamId={team_id}",
                })
                continue

            outgoing_found = (out_id is not None) and (out_id in lineup)
            incoming_already = (in_id is not None) and (in_id in lineup)
            can_apply = outgoing_found and (not incoming_already) and (in_id is not None)

            if can_apply:
                lineup.remove(out_id)
                lineup.add(in_id)

            size_ok = (len(home) == 5) and (len(away) == 5)
            err = None
            if not outgoing_found:
                err = "outgoing_not_found"
            elif incoming_already:
                err = "incoming_already_present"
            elif in_id is None:
                err = "incoming_id_missing"

            rows.append({
                "period": period,
                "clock": clock,
                "team": team,
                "out_id": out_id,
                "in_id": in_id,
                "outgoing_found": outgoing_found,
                "incoming_already_present": incoming_already,
                "size_ok": size_ok,
                "applied": can_apply,
                "error": err,
            })

    val_df = pd.DataFrame(rows)
    if len(val_df) == 0:
        validation_pass = True
    else:
        validation_pass = bool(
            val_df["outgoing_found"].all()
            and (~val_df["incoming_already_present"]).all()
            and val_df["size_ok"].all()
        )

    return {
        "period": period,
        "n_batches_checked": int(min(n_batches, len(q_subs))),
        "validation_pass": validation_pass,
        "rows": rows,
        "home_final": home,
        "away_final": away,
    }


validation_v2 = {}
for period in [2, 3, 4]:
    res = validate_n_sub_batches(
        period=period,
        home_open_ids=period_start_lineups_inferred[period]["home"],
        away_open_ids=period_start_lineups_inferred[period]["away"],
        sub_groups_df=sub_groups_df,
        sub_group_lookup=sub_group_lookup,
        n_batches=3,
    )
    validation_v2[period] = res

    print(f"\n===== STRONG VALIDATION P{period} =====")
    print("Batches checked:", res["n_batches_checked"])
    print("PASS:", res["validation_pass"])

    if len(res["rows"]):
        d = pd.DataFrame(res["rows"])
        d["out_name"] = d["out_id"].apply(lambda x: player_id_to_name.get(int(x), f"id:{int(x)}") if pd.notna(x) else None) if "player_id_to_name" in globals() else d["out_id"]
        d["in_name"] = d["in_id"].apply(lambda x: player_id_to_name.get(int(x), f"id:{int(x)}") if pd.notna(x) else None) if "player_id_to_name" in globals() else d["in_id"]
        display(d)

        failed = d[(~d["outgoing_found"]) | (d["incoming_already_present"]) | (~d["size_ok"])]
        if len(failed):
            print("Failures:")
            display(failed)
        else:
            print("All checked substitutions passed validation.")


===== STRONG VALIDATION P2 =====
Batches checked: 3
PASS: True


,period,clock,team,out_id,in_id,outgoing_found,incoming_already_present,size_ok,applied,error,out_name,in_name
0,2,PT08M43.00S,HOME,1630568,1628401,True,False,True,True,None,Garza,White
1,2,PT08M43.00S,HOME,1631248,1629674,True,False,True,True,None,Scheierman,Queta
2,2,PT08M43.00S,AWAY,1629656,1630178,True,False,True,True,None,Grimes,Maxey
3,2,PT07M16.00S,HOME,1630573,1628369,True,False,True,True,None,Hauser,Tatum
4,2,PT07M16.00S,AWAY,203083,1641737,True,False,True,True,None,Drummond,Bona
5,2,PT06M50.00S,AWAY,202331,1626162,True,False,True,True,None,George,Oubre Jr.


All checked substitutions passed validation.

===== STRONG VALIDATION P3 =====
Batches checked: 3
PASS: True


,period,clock,team,out_id,in_id,outgoing_found,incoming_already_present,size_ok,applied,error,out_name,in_name
0,3,PT10M25.00S,HOME,1629674,202696,True,False,True,True,None,Queta,Vučević
1,3,PT06M45.00S,HOME,1630573,1630202,True,False,True,True,None,Hauser,Pritchard
2,3,PT04M33.00S,AWAY,203083,1629656,True,False,True,True,None,Drummond,Grimes
3,3,PT04M33.00S,AWAY,1626162,1641737,True,False,True,True,None,Oubre Jr.,Bona


All checked substitutions passed validation.

===== STRONG VALIDATION P4 =====
Batches checked: 3
PASS: True


,period,clock,team,out_id,in_id,outgoing_found,incoming_already_present,size_ok,applied,error,out_name,in_name
0,4,PT09M16.00S,AWAY,1642348,203083,True,False,True,True,None,Edwards,Drummond
1,4,PT09M16.00S,AWAY,1631230,1626162,True,False,True,True,None,Barlow,Oubre Jr.
2,4,PT08M11.00S,AWAY,1630178,1642348,True,False,True,True,None,Maxey,Edwards
3,4,PT07M15.00S,HOME,1628369,1641775,True,False,True,True,None,Tatum,Walsh


All checked substitutions passed validation.


In [38]:
# Cell 1: Final stint-builder helpers (ID-first)


def seconds_from_clock(clock):
    return float(_clock_to_seconds(clock)) if _clock_to_seconds(clock) is not None else None


def get_score_snapshot_for_period_clock(pbp_df, period, clock, last_known_score=(0, 0)):
    """Return score state at this period+clock, with safe carry-forward fallback."""
    rows = pbp_df[(pbp_df["period"].eq(period)) & (pbp_df["clock"].eq(clock))].copy()
    if rows.empty:
        return last_known_score

    sh = pd.to_numeric(rows.get("scoreHome"), errors="coerce")
    sa = pd.to_numeric(rows.get("scoreAway"), errors="coerce")

    # Take latest non-null score in this group; otherwise carry forward.
    sh_nonnull = sh.dropna()
    sa_nonnull = sa.dropna()

    score_home = int(sh_nonnull.iloc[-1]) if len(sh_nonnull) else int(last_known_score[0])
    score_away = int(sa_nonnull.iloc[-1]) if len(sa_nonnull) else int(last_known_score[1])
    return score_home, score_away


def build_stint_row(
    game_id,
    period,
    start_clock,
    end_clock,
    home_team_id,
    away_team_id,
    home_lineup_ids,
    away_lineup_ids,
    score_home_start,
    score_away_start,
    score_home_end,
    score_away_end,
    ended_by_substitution,
):
    start_s = seconds_from_clock(start_clock)
    end_s = seconds_from_clock(end_clock)
    duration = None if (start_s is None or end_s is None) else float(start_s - end_s)

    home_ids_sorted = sorted(int(i) for i in home_lineup_ids)
    away_ids_sorted = sorted(int(i) for i in away_lineup_ids)

    return {
        "game_id": str(game_id),
        "period": int(period),
        "start_clock": str(start_clock),
        "end_clock": str(end_clock),
        "start_seconds_remaining": start_s,
        "end_seconds_remaining": end_s,
        "duration_seconds": duration,
        "home_team_id": int(home_team_id),
        "away_team_id": int(away_team_id),
        "home_lineup_ids": home_ids_sorted,
        "away_lineup_ids": away_ids_sorted,
        "home_lineup_names": [player_id_to_name.get(i, f"id:{i}") for i in home_ids_sorted] if "player_id_to_name" in globals() else home_ids_sorted,
        "away_lineup_names": [player_id_to_name.get(i, f"id:{i}") for i in away_ids_sorted] if "player_id_to_name" in globals() else away_ids_sorted,
        "score_home_start": int(score_home_start),
        "score_away_start": int(score_away_start),
        "score_home_end": int(score_home_end),
        "score_away_end": int(score_away_end),
        "ended_by_substitution": bool(ended_by_substitution),
    }

In [39]:
# Cell 2: Build final one-game fact_lineup_stint from validated period starts

# Regulation periods only (1-4). Keep grouped timestamp workflow unchanged.
regulation_periods = [1, 2, 3, 4]

stint_rows_final = []
last_known_score = (0, 0)

for period in regulation_periods:
    if period not in period_start_lineups_inferred:
        raise ValueError(f"Missing period_start_lineups_inferred for period {period}")

    current_home_lineup_ids = set(period_start_lineups_inferred[period]["home"])
    current_away_lineup_ids = set(period_start_lineups_inferred[period]["away"])

    if len(current_home_lineup_ids) != 5 or len(current_away_lineup_ids) != 5:
        raise ValueError(f"Period {period} opening lineups must each have 5 players")

    period_start_clock = "PT12M00.00S"
    period_end_clock = "PT00M00.00S"

    # Score at period start group, with carry-forward safety.
    start_score_home, start_score_away = get_score_snapshot_for_period_clock(
        pbp_df,
        period,
        period_start_clock,
        last_known_score=last_known_score,
    )

    current_stint_start_clock = period_start_clock
    current_stint_score_home_start = start_score_home
    current_stint_score_away_start = start_score_away

    # Iterate timestamp groups in game order within period (high -> low seconds remaining).
    period_groups = (
        event_groups_df[event_groups_df["period"].eq(period)]
        .sort_values("clock_seconds_remaining", ascending=False)
        .reset_index(drop=True)
    )

    for _, g in period_groups.iterrows():
        clock = g["clock"]
        has_sub = bool(g["has_substitution"])

        score_home_here, score_away_here = get_score_snapshot_for_period_clock(
            pbp_df,
            period,
            clock,
            last_known_score=last_known_score,
        )
        last_known_score = (score_home_here, score_away_here)

        if not has_sub:
            continue

        # Close stint at substitution boundary.
        stint_rows_final.append(
            build_stint_row(
                game_id=GAME_ID,
                period=period,
                start_clock=current_stint_start_clock,
                end_clock=clock,
                home_team_id=home_team_id,
                away_team_id=away_team_id,
                home_lineup_ids=current_home_lineup_ids,
                away_lineup_ids=current_away_lineup_ids,
                score_home_start=current_stint_score_home_start,
                score_away_start=current_stint_score_away_start,
                score_home_end=score_home_here,
                score_away_end=score_away_here,
                ended_by_substitution=True,
            )
        )

        # Apply full same-clock substitution batch, then open next stint at same clock.
        sub_batch = sub_group_lookup.get((period, clock), [])
        current_home_lineup_ids, current_away_lineup_ids, _ = apply_sub_batch_ids(
            current_home_lineup_ids,
            current_away_lineup_ids,
            sub_batch,
            home_team_id=home_team_id,
            away_team_id=away_team_id,
        )

        current_stint_start_clock = clock
        current_stint_score_home_start = score_home_here
        current_stint_score_away_start = score_away_here

    # Flush final open stint to end of period at PT00M00.00S.
    period_end_score_home, period_end_score_away = get_score_snapshot_for_period_clock(
        pbp_df,
        period,
        period_end_clock,
        last_known_score=last_known_score,
    )
    last_known_score = (period_end_score_home, period_end_score_away)

    stint_rows_final.append(
        build_stint_row(
            game_id=GAME_ID,
            period=period,
            start_clock=current_stint_start_clock,
            end_clock=period_end_clock,
            home_team_id=home_team_id,
            away_team_id=away_team_id,
            home_lineup_ids=current_home_lineup_ids,
            away_lineup_ids=current_away_lineup_ids,
            score_home_start=current_stint_score_home_start,
            score_away_start=current_stint_score_away_start,
            score_home_end=period_end_score_home,
            score_away_end=period_end_score_away,
            ended_by_substitution=False,
        )
    )

fact_lineup_stint = pd.DataFrame(stint_rows_final)
print("fact_lineup_stint rows:", len(fact_lineup_stint))

fact_lineup_stint rows: 35


In [40]:
# Cell 3: Display first stints with readable lineups

display_cols = [
    "game_id", "period", "start_clock", "end_clock", "duration_seconds",
    "home_lineup_names", "away_lineup_names",
    "score_home_start", "score_away_start", "score_home_end", "score_away_end",
    "ended_by_substitution",
]

display(fact_lineup_stint[display_cols].head(20))

,game_id,period,start_clock,end_clock,duration_seconds,home_lineup_names,away_lineup_names,score_home_start,score_away_start,score_home_end,score_away_end,ended_by_substitution
0,0042500111,1,PT12M00.00S,PT10M23.00S,97.0,"[Brown, Tatum, White, Queta, Hauser]","[George, Oubre Jr., Maxey, Bona, Edgecombe]",0,0,4,3,True
1,0042500111,1,PT10M23.00S,PT08M39.00S,104.0,"[Brown, Tatum, White, Queta, Hauser]","[George, Drummond, Oubre Jr., Maxey, Edgecombe]",4,3,8,7,True
2,0042500111,1,PT08M39.00S,PT04M50.00S,229.0,"[Vučević, Brown, Tatum, White, Hauser]","[George, Drummond, Oubre Jr., Maxey, Edgecombe]",8,7,22,11,True
3,0042500111,1,PT04M50.00S,PT04M39.00S,11.0,"[Vučević, Brown, Tatum, White, Pritchard]","[George, Drummond, Grimes, Maxey, Barlow]",22,11,22,11,True
4,0042500111,1,PT04M39.00S,PT03M49.00S,50.0,"[Vučević, Brown, Tatum, White, Pritchard]","[George, Oubre Jr., Grimes, Maxey, Barlow]",22,11,22,14,True
5,0042500111,1,PT03M49.00S,PT02M39.00S,70.0,"[Vučević, Tatum, White, Pritchard, Walsh]","[George, Oubre Jr., Grimes, Maxey, Barlow]",22,14,24,16,True
6,0042500111,1,PT02M39.00S,PT00M27.30S,131.7,"[Vučević, Tatum, White, Pritchard, Walsh]","[Oubre Jr., Grimes, Maxey, Barlow, Edwards]",24,16,31,18,True
7,0042500111,1,PT00M27.30S,PT00M00.00S,27.3,"[Brown, Tatum, White, Pritchard, Walsh]","[George, Oubre Jr., Grimes, Maxey, Edwards]",31,18,33,18,False
8,0042500111,2,PT12M00.00S,PT08M43.00S,197.0,"[Brown, Pritchard, Garza, Hauser, Scheierman]","[George, Drummond, Grimes, Edwards, Edgecombe]",33,18,42,27,True
9,0042500111,2,PT08M43.00S,PT07M16.00S,87.0,"[Brown, White, Queta, Pritchard, Hauser]","[George, Drummond, Maxey, Edwards, Edgecombe]",42,27,49,29,True


In [41]:
# Cell 4: Validation checks for final one-game stint table

# 1) Duration checks
neg_dur = fact_lineup_stint[fact_lineup_stint["duration_seconds"] < 0]
zero_dur = fact_lineup_stint[fact_lineup_stint["duration_seconds"] == 0]

# 2) Lineup-size checks
bad_home_size = fact_lineup_stint[fact_lineup_stint["home_lineup_ids"].apply(lambda x: len(set(x)) != 5)]
bad_away_size = fact_lineup_stint[fact_lineup_stint["away_lineup_ids"].apply(lambda x: len(set(x)) != 5)]

# 3) Regulation total duration check (4 * 720 = 2880)
total_duration = float(fact_lineup_stint["duration_seconds"].sum())

# 4) Substitution-boundary sanity:
#    if lineup changes between consecutive stints in same period,
#    prior stint should end by substitution and boundary clock should match.
f = fact_lineup_stint.sort_values(["period", "start_seconds_remaining", "end_seconds_remaining"], ascending=[True, False, False]).reset_index(drop=True)

boundary_issues = []
for i in range(len(f) - 1):
    cur = f.iloc[i]
    nxt = f.iloc[i + 1]

    if int(cur["period"]) != int(nxt["period"]):
        continue

    cur_home = set(cur["home_lineup_ids"])
    cur_away = set(cur["away_lineup_ids"])
    nxt_home = set(nxt["home_lineup_ids"])
    nxt_away = set(nxt["away_lineup_ids"])

    lineup_changed = (cur_home != nxt_home) or (cur_away != nxt_away)
    if not lineup_changed:
        continue

    boundary_ok = bool(cur["ended_by_substitution"]) and (str(cur["end_clock"]) == str(nxt["start_clock"]))
    if not boundary_ok:
        boundary_issues.append({
            "idx": i,
            "period": int(cur["period"]),
            "cur_end_clock": cur["end_clock"],
            "next_start_clock": nxt["start_clock"],
            "cur_ended_by_substitution": bool(cur["ended_by_substitution"]),
            "cur_home": cur["home_lineup_names"],
            "nxt_home": nxt["home_lineup_names"],
            "cur_away": cur["away_lineup_names"],
            "nxt_away": nxt["away_lineup_names"],
        })

boundary_issues_df = pd.DataFrame(boundary_issues)

print("Negative durations:", len(neg_dur))
print("Zero-duration stints:", len(zero_dur))
print("Bad home lineup size (!=5):", len(bad_home_size))
print("Bad away lineup size (!=5):", len(bad_away_size))
print("Total duration seconds:", total_duration)
print("Expected regulation total:", 2880.0)
print("Duration total check pass:", abs(total_duration - 2880.0) < 1e-9)
print("Boundary sanity issues:", len(boundary_issues_df))

if len(neg_dur):
    print("\nNegative duration rows:")
    display(neg_dur)
if len(zero_dur):
    print("\nZero duration rows:")
    display(zero_dur)
if len(bad_home_size):
    print("\nBad home lineup size rows:")
    display(bad_home_size)
if len(bad_away_size):
    print("\nBad away lineup size rows:")
    display(bad_away_size)
if len(boundary_issues_df):
    print("\nBoundary issues:")
    display(boundary_issues_df)

Negative durations: 0
Zero-duration stints: 0
Bad home lineup size (!=5): 0
Bad away lineup size (!=5): 0
Total duration seconds: 2880.0
Expected regulation total: 2880.0
Duration total check pass: True
Boundary sanity issues: 0


In [42]:
# Cell 5: Short explanation of builder + validations

print(
    """
How this final stint builder works
- Starts each period from validated period_start_lineups_inferred.
- Opens a stint at PT12M00.00S with period-start score snapshot.
- Walks grouped timestamps in game order (descending clock within period).
- At each substitution timestamp: closes current stint, applies full same-clock sub batch, opens next stint at same clock.
- Flushes final open stint to PT00M00.00S using the period's last known score state.

What validations check
1) Duration validity: no negative and no zero-duration stints.
2) Lineup integrity: home/away lineup IDs each contain exactly 5 unique players.
3) Coverage: total regulation duration sums to 2880 seconds.
4) Boundary sanity: lineup changes only occur where prior stint ended by substitution and clocks align (end == next start).
"""
)


How this final stint builder works
- Starts each period from validated period_start_lineups_inferred.
- Opens a stint at PT12M00.00S with period-start score snapshot.
- Walks grouped timestamps in game order (descending clock within period).
- At each substitution timestamp: closes current stint, applies full same-clock sub batch, opens next stint at same clock.
- Flushes final open stint to PT00M00.00S using the period's last known score state.

What validations check
1) Duration validity: no negative and no zero-duration stints.
2) Lineup integrity: home/away lineup IDs each contain exactly 5 unique players.
3) Coverage: total regulation duration sums to 2880 seconds.
4) Boundary sanity: lineup changes only occur where prior stint ended by substitution and clocks align (end == next start).



In [43]:
# Demo cell: End-to-end walkthrough of how lineup stints were built
# Share this cell with a friend to explain the full workflow quickly.

print("=" * 72)
print("NBA ONE-GAME LINEUP STINT PIPELINE (ID-FIRST)")
print("=" * 72)

print("\n1) Inputs loaded")
print("- play-by-play actions: pbp_df")
print("- grouped timestamps: event_groups_df")
print("- grouped same-clock substitutions: sub_groups_df / sub_group_lookup")
print("- player lookup: player_id_to_name (for readable display only)")

print("\n2) Quarter-opening lineup inference")
print("- Q1 from official live boxscore starters")
print("- Q2/Q3/Q4 inferred from game data (player IDs):")
print("  * first substitution role by player in quarter (first OUT => started)")
print("  * early-quarter real-event player appearances")
print("  * fallback prior (Q3 uses Q1 starters; Q2/Q4 use prior period end)")
print("  * exclude players whose first role in quarter is IN")

print("\n3) Inference validation")
print("- Applied first 2-3 substitution batches per period")
print("- Checked:")
print("  * outgoing player is currently on floor")
print("  * incoming player is not already on floor")
print("  * lineup size remains exactly 5 per team")

print("\n4) Final fact_lineup_stint construction")
print("- For each period:")
print("  * initialize from period_start_lineups_inferred")
print("  * open stint at PT12M00.00S")
print("  * close/open stints only at substitution timestamps")
print("  * apply full same-clock substitution batch")
print("  * flush final stint to PT00M00.00S")
print("- Score start/end captured from grouped timestamp snapshots")

print("\n5) Final table quality checks")
if "fact_lineup_stint" in globals() and len(fact_lineup_stint):
    neg = int((fact_lineup_stint["duration_seconds"] < 0).sum())
    zero = int((fact_lineup_stint["duration_seconds"] == 0).sum())
    bad_home = int(fact_lineup_stint["home_lineup_ids"].apply(lambda x: len(set(x)) != 5).sum())
    bad_away = int(fact_lineup_stint["away_lineup_ids"].apply(lambda x: len(set(x)) != 5).sum())
    total_sec = float(fact_lineup_stint["duration_seconds"].sum())

    print(f"- rows: {len(fact_lineup_stint)}")
    print(f"- negative durations: {neg}")
    print(f"- zero durations: {zero}")
    print(f"- bad home lineup sizes: {bad_home}")
    print(f"- bad away lineup sizes: {bad_away}")
    print(f"- total regulation seconds: {total_sec} (target: 2880)")

    print("\nSample stints:")
    display(
        fact_lineup_stint[
            [
                "period", "start_clock", "end_clock", "duration_seconds",
                "home_lineup_names", "away_lineup_names",
                "score_home_start", "score_away_start", "score_home_end", "score_away_end",
                "ended_by_substitution",
            ]
        ].head(10)
    )
else:
    print("- fact_lineup_stint not found yet. Run final build cells first.")

print("\nDone: This creates an auditable, modeling-ready one-game lineup stint table.")

NBA ONE-GAME LINEUP STINT PIPELINE (ID-FIRST)

1) Inputs loaded
- play-by-play actions: pbp_df
- grouped timestamps: event_groups_df
- grouped same-clock substitutions: sub_groups_df / sub_group_lookup
- player lookup: player_id_to_name (for readable display only)

2) Quarter-opening lineup inference
- Q1 from official live boxscore starters
- Q2/Q3/Q4 inferred from game data (player IDs):
  * first substitution role by player in quarter (first OUT => started)
  * early-quarter real-event player appearances
  * fallback prior (Q3 uses Q1 starters; Q2/Q4 use prior period end)
  * exclude players whose first role in quarter is IN

3) Inference validation
- Applied first 2-3 substitution batches per period
- Checked:
  * outgoing player is currently on floor
  * incoming player is not already on floor
  * lineup size remains exactly 5 per team

4) Final fact_lineup_stint construction
- For each period:
  * initialize from period_start_lineups_inferred
  * open stint at PT12M00.00S
  * clo

,period,start_clock,end_clock,duration_seconds,home_lineup_names,away_lineup_names,score_home_start,score_away_start,score_home_end,score_away_end,ended_by_substitution
0,1,PT12M00.00S,PT10M23.00S,97.0,"[Brown, Tatum, White, Queta, Hauser]","[George, Oubre Jr., Maxey, Bona, Edgecombe]",0,0,4,3,True
1,1,PT10M23.00S,PT08M39.00S,104.0,"[Brown, Tatum, White, Queta, Hauser]","[George, Drummond, Oubre Jr., Maxey, Edgecombe]",4,3,8,7,True
2,1,PT08M39.00S,PT04M50.00S,229.0,"[Vučević, Brown, Tatum, White, Hauser]","[George, Drummond, Oubre Jr., Maxey, Edgecombe]",8,7,22,11,True
3,1,PT04M50.00S,PT04M39.00S,11.0,"[Vučević, Brown, Tatum, White, Pritchard]","[George, Drummond, Grimes, Maxey, Barlow]",22,11,22,11,True
4,1,PT04M39.00S,PT03M49.00S,50.0,"[Vučević, Brown, Tatum, White, Pritchard]","[George, Oubre Jr., Grimes, Maxey, Barlow]",22,11,22,14,True
5,1,PT03M49.00S,PT02M39.00S,70.0,"[Vučević, Tatum, White, Pritchard, Walsh]","[George, Oubre Jr., Grimes, Maxey, Barlow]",22,14,24,16,True
6,1,PT02M39.00S,PT00M27.30S,131.7,"[Vučević, Tatum, White, Pritchard, Walsh]","[Oubre Jr., Grimes, Maxey, Barlow, Edwards]",24,16,31,18,True
7,1,PT00M27.30S,PT00M00.00S,27.3,"[Brown, Tatum, White, Pritchard, Walsh]","[George, Oubre Jr., Grimes, Maxey, Edwards]",31,18,33,18,False
8,2,PT12M00.00S,PT08M43.00S,197.0,"[Brown, Pritchard, Garza, Hauser, Scheierman]","[George, Drummond, Grimes, Edwards, Edgecombe]",33,18,42,27,True
9,2,PT08M43.00S,PT07M16.00S,87.0,"[Brown, White, Queta, Pritchard, Hauser]","[George, Drummond, Maxey, Edwards, Edgecombe]",42,27,49,29,True



Done: This creates an auditable, modeling-ready one-game lineup stint table.
